In [2]:
import pandas as pd
from pathlib import Path

CLEAN_DATA_PATH = Path("../Data/Cleaned")

files = [
    "closed_deals.csv",
    "customers.csv",
    "geolocation.csv",
    "marketing_qualified_leads.csv",
    "order_items.csv",
    "order_payments.csv",
    "order_reviews.csv",
    "orders.csv",
    "products.csv",
    "sellers.csv",
    "product_category_name_translation.csv"
]

data = {}

for file in files:
    dataset_name = Path(file).stem
    data[dataset_name] = pd.read_csv(CLEAN_DATA_PATH / file)

print("=" * 70)
print("ShopSphere Business Analysis")
print("=" * 70)

for name, df in data.items():
    print(f"{name:<40} {df.shape}")

print("=" * 70)
print("All cleaned datasets loaded successfully.")

ShopSphere Business Analysis
closed_deals                             (842, 15)
customers                                (99441, 5)
geolocation                              (1000163, 5)
marketing_qualified_leads                (8000, 4)
order_items                              (112650, 7)
order_payments                           (103886, 5)
order_reviews                            (99224, 7)
orders                                   (99441, 8)
products                                 (32951, 9)
sellers                                  (3095, 4)
product_category_name_translation        (71, 2)
All cleaned datasets loaded successfully.


# ShopSphere Business Analysis

## Project Objective

Analyze ShopSphere's sales, customer, product, seller, payment,
delivery, satisfaction, and marketing data to identify key business
performance patterns and actionable opportunities.

## Analytical Approach

The analysis progresses from data validation and exploratory analysis
to order-level modeling, KPI construction, business insights,
scenario analysis, and final quality assurance.

## Key Business Questions

### 1. Sales Performance
- How is revenue changing over time?
- Which months generate the highest sales?
- What is the average order value?

### 2. Product Performance
- Which product categories generate the most revenue?
- Which categories sell the most items?
- Which categories have the highest average prices?

### 3. Customer Analysis
- Which states generate the most customers and orders?
- Which locations generate the highest revenue?
- How many repeat customers does the business have?

### 4. Seller Analysis
- Who are the top-performing sellers?
- Which states have the highest number of sellers?
- Which sellers generate the most revenue?

### 5. Payment Analysis
- Which payment methods are most commonly used?
- What is the relationship between payment installments and order value?

### 6. Delivery Performance
- What is the average delivery time?
- How many orders are delivered late?
- Which states or categories experience longer delivery times?

### 7. Customer Satisfaction
- What is the overall average review score?
- Does delayed delivery affect customer reviews?
- Which categories have the lowest customer satisfaction?

# 1. Data Structure & Relationship Validation

Before performing business analysis, it is important to understand how the
datasets are structured and how they relate to one another.

The main transaction tables do not all have the same grain:

- `orders` contains one row per order.
- `order_items` contains multiple rows per order.
- `order_payments` can contain multiple payment records per order.
- `order_reviews` can contain multiple reviews per order.

This section validates table granularity, duplicate structure, relationship
coverage, and missing relationships before any aggregation or joining is
performed.

This prevents double-counting and incorrect business metrics later.

In [3]:
# Define the primary identifier for each dataset
dataset_keys = {
    "orders": "order_id",
    "order_items": "order_id",
    "order_payments": "order_id",
    "order_reviews": "order_id",
    "customers": "customer_id",
    "products": "product_id",
    "sellers": "seller_id",
    "product_category_name_translation": "product_category_name"
}

for dataset, key in dataset_keys.items():
    
    df = data[dataset]
    
    total_rows = len(df)
    unique_keys = df[key].nunique()
    
    print(f"\n{dataset.upper()}")
    print(f"Total rows       : {total_rows:,}")
    print(f"Unique {key:<13}: {unique_keys:,}")
    
    if total_rows == unique_keys:
        print("Granularity      : ONE ROW PER KEY")
    else:
        print(f"Granularity      : MULTIPLE ROWS PER {key.upper()}")


ORDERS
Total rows       : 99,441
Unique order_id     : 99,441
Granularity      : ONE ROW PER KEY

ORDER_ITEMS
Total rows       : 112,650
Unique order_id     : 98,666
Granularity      : MULTIPLE ROWS PER ORDER_ID

ORDER_PAYMENTS
Total rows       : 103,886
Unique order_id     : 99,440
Granularity      : MULTIPLE ROWS PER ORDER_ID

ORDER_REVIEWS
Total rows       : 99,224
Unique order_id     : 98,673
Granularity      : MULTIPLE ROWS PER ORDER_ID

CUSTOMERS
Total rows       : 99,441
Unique customer_id  : 99,441
Granularity      : ONE ROW PER KEY

PRODUCTS
Total rows       : 32,951
Unique product_id   : 32,951
Granularity      : ONE ROW PER KEY

SELLERS
Total rows       : 3,095
Unique seller_id    : 3,095
Granularity      : ONE ROW PER KEY

PRODUCT_CATEGORY_NAME_TRANSLATION
Total rows       : 71
Unique product_category_name: 71
Granularity      : ONE ROW PER KEY


In [4]:
print("STEP 2B — ORDER-LEVEL DUPLICATE STRUCTURE")

tables_to_check = [
    "order_items",
    "order_payments",
    "order_reviews"
]

for dataset in tables_to_check:
    
    df = data[dataset]
    
    counts = df.groupby("order_id").size()
    
    print(f"\n{dataset.upper()}")
    print(f"Orders represented     : {counts.shape[0]:,}")
    print(f"Minimum records/order  : {counts.min()}")
    print(f"Maximum records/order  : {counts.max()}")
    print(f"Average records/order  : {counts.mean():.2f}")
    
    print("\nDistribution:")
    print(counts.value_counts().sort_index().head(15))

STEP 2B — ORDER-LEVEL DUPLICATE STRUCTURE

ORDER_ITEMS
Orders represented     : 98,666
Minimum records/order  : 1
Maximum records/order  : 21
Average records/order  : 1.14

Distribution:
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
Name: count, dtype: int64

ORDER_PAYMENTS
Orders represented     : 99,440
Minimum records/order  : 1
Maximum records/order  : 29
Average records/order  : 1.04

Distribution:
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
Name: count, dtype: int64

ORDER_REVIEWS
Orders represented     : 98,673
Minimum records/order  : 1
Maximum records/order  : 3
Average records/order  : 1.01

Distribution:
1    98126
2      543
3        4
Name: count, dtype: int64


In [5]:
print("STEP 2C — MULTIPLE REVIEW INVESTIGATION")

reviews = data["order_reviews"]

# Orders with more than one review record
review_counts = reviews.groupby("order_id").size()

multi_review_orders = review_counts[
    review_counts > 1
].index

multi_reviews = reviews[
    reviews["order_id"].isin(multi_review_orders)
].sort_values(["order_id", "review_creation_date"])

print(f"Orders with multiple review records: {len(multi_review_orders):,}")
print(f"Total review records involved: {len(multi_reviews):,}")

print("\nDuplicate review IDs within the same order:")
duplicate_review_ids = multi_reviews.duplicated(
    subset=["order_id", "review_id"],
    keep=False
).sum()

print(duplicate_review_ids)

print("\nSample orders with multiple reviews:")
display(
    multi_reviews[
        [
            "order_id",
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].head(20)
)

STEP 2C — MULTIPLE REVIEW INVESTIGATION


Orders with multiple review records: 547
Total review records involved: 1,098

Duplicate review IDs within the same order:
0

Sample orders with multiple reviews:


,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04 00:00:00,2018-03-05 17:02:00
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2017-12-30 00:00:00,2018-01-02 10:54:47
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-21 00:00:00,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-29 00:00:00,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,2017-07-19 00:00:00,2017-07-20 12:06:11


In [6]:
# ============================================================
# STEP 2C — MULTIPLE REVIEW INVESTIGATION
# ============================================================

multiple_reviews = (
    data["order_reviews"]
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        unique_review_ids=("review_id", "nunique"),
        average_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max")
    )
    .reset_index()
)

# Keep only orders with multiple review records
multiple_reviews = multiple_reviews[
    multiple_reviews["review_count"] > 1
].copy()

print("Orders with multiple reviews:", len(multiple_reviews))

print("\nDistribution of reviews per order:")
print(
    multiple_reviews["review_count"]
    .value_counts()
    .sort_index()
)

print("\nSample:")
display(multiple_reviews.head(20))

Orders with multiple reviews: 547

Distribution of reviews per order:
review_count
2    543
3      4
Name: count, dtype: int64

Sample:


,order_id,review_count,unique_review_ids,average_review_score,min_review_score,max_review_score
84,0035246a40f520710769010f752e7507,2,2,5.000000,5,5
461,013056cfe49763c6f66bda03396c5ee3,2,2,4.500000,4,5
556,0176a6846bcb3b0d3aa3116a9a768597,2,2,5.000000,5,5
835,02355020fd0a40a0d56df9f6ff060413,2,2,2.000000,1,3
985,029863af4b968de1e5d6a82782e662f5,2,2,4.500000,4,5
1092,02e0b68852217f5715fb9cc885829454,2,2,4.000000,4,4
1103,02e723e8edb4a123d414f56cc9c4665e,2,2,5.000000,5,5
1272,03515a836bb855b03f7df9dee520a8fc,2,2,5.000000,5,5
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,3,3.333333,3,4
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,4.500000,4,5


In [7]:
# STEP 2D — PAYMENT STRUCTURE ANALYSIS

payment_analysis = (
    data["order_payments"]
    .groupby("order_id")
    .agg(
        payment_records=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        unique_payment_types=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
)

multiple_payments = payment_analysis[
    payment_analysis["payment_records"] > 1
]

print("Orders with multiple payment records:", len(multiple_payments))

print("\nDistribution of payment records per order:")
print(
    multiple_payments["payment_records"]
    .value_counts()
    .sort_index()
)

print("\nSample of orders with multiple payments:")
display(multiple_payments.head(20))

Orders with multiple payment records: 2961

Distribution of payment records per order:
payment_records
2     2382
3      301
4      108
5       52
6       36
7       28
8       11
9        9
10       5
11       8
12       8
13       3
14       2
15       2
19       2
21       1
22       1
26       1
29       1
Name: count, dtype: int64

Sample of orders with multiple payments:


,payment_records,total_payment_value,unique_payment_types,max_installments
order_id,,,,
0016dfedd97fc2950e388d2971d718c7,2,70.55,2,5
002f19a65a2ddd70a090297872e6d64e,2,77.29,1,1
0071ee2429bc1efdc43aa3e073a5290e,2,192.44,1,1
009ac365164f8e06f59d18a08045f6c4,6,32.00,2,1
00b4a910f64f24dbcac04fe54088a443,2,50.59,2,1
00bd50cdd31bd22e9081e6e2d5b3577b,3,85.80,2,1
00c405bd71187154a7846862f585a9d4,7,46.69,2,1
00c95282163553a982f38481f9488481,4,124.90,2,1
00e6bc6b166eb28b4502c1cad4457248,2,151.93,2,2


In [8]:
# STEP 2E — ORDER ITEMS STRUCTURE ANALYSIS

item_analysis = (
    data["order_items"]
    .groupby("order_id")
    .agg(
        item_records=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        total_product_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
)

multiple_items = item_analysis[
    item_analysis["item_records"] > 1
]

print("Orders with multiple item records:", len(multiple_items))

print("\nDistribution of item records per order:")
print(
    multiple_items["item_records"]
    .value_counts()
    .sort_index()
)

print("\nSample of orders with multiple items:")
display(multiple_items.head(20))

Orders with multiple item records: 9803

Distribution of item records per order:
item_records
2     7516
3     1322
4      505
5      204
6      198
7       22
8        8
9        3
10       8
11       4
12       5
13       1
14       2
15       2
20       2
21       1
Name: count, dtype: int64

Sample of orders with multiple items:


,item_records,unique_products,unique_sellers,total_product_value,total_freight_value
order_id,,,,,
0008288aa423d2a3f00fcb17cd7d8719,2,1,1,99.80,26.74
00143d0f86d6fbd9f9b38ab440ac16f5,3,1,1,63.99,45.30
001ab0a7578dd66cd4b0a71f5b6e1e41,3,1,1,74.67,52.89
001d8f0e34a38c37f7dba2a37d4eba8b,2,1,1,37.98,15.56
002c9def9c9b951b1bec6d50753c9891,2,1,1,156.00,17.80
002f98c0f7efd42638ed6100ca699b42,2,2,2,53.89,39.73
003324c70b19a16798817b2b3640e721,2,1,1,205.80,28.90
00337fe25a3780b3424d9ad7c5a4b35e,2,2,1,119.80,19.88
003822434f91204da0a51fe4cf2aba18,2,1,1,138.00,37.16


In [9]:
# STEP 3 — ORDER RELATIONSHIP COVERAGE ANALYSIS

orders = data["orders"]

order_ids = set(orders["order_id"])

items_order_ids = set(data["order_items"]["order_id"])
payments_order_ids = set(data["order_payments"]["order_id"])
reviews_order_ids = set(data["order_reviews"]["order_id"])

print("STEP 3 — ORDER RELATIONSHIP COVERAGE")

print("\nTotal orders:", len(order_ids))

print("\nOrders WITHOUT items:")
missing_items = order_ids - items_order_ids
print("Count:", len(missing_items))

print("\nOrders WITHOUT payments:")
missing_payments = order_ids - payments_order_ids
print("Count:", len(missing_payments))

print("\nOrders WITHOUT reviews:")
missing_reviews = order_ids - reviews_order_ids
print("Count:", len(missing_reviews))


print("\nSTATUS BREAKDOWN FOR ORDERS WITH MISSING RELATIONSHIPS")


for name, missing_ids in {
    "Missing Items": missing_items,
    "Missing Payments": missing_payments,
    "Missing Reviews": missing_reviews
}.items():
    
    print(f"\n{name}:")
    
    if len(missing_ids) > 0:
        print(
            orders[
                orders["order_id"].isin(missing_ids)
            ]["order_status"].value_counts()
        )
    else:
        print("None")

STEP 3 — ORDER RELATIONSHIP COVERAGE

Total orders: 99441

Orders WITHOUT items:
Count: 775

Orders WITHOUT payments:
Count: 1

Orders WITHOUT reviews:
Count: 768

STATUS BREAKDOWN FOR ORDERS WITH MISSING RELATIONSHIPS

Missing Items:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Missing Payments:
order_status
delivered    1
Name: count, dtype: int64

Missing Reviews:
order_status
delivered      646
shipped         75
canceled        20
unavailable     14
processing       6
invoiced         5
created          2
Name: count, dtype: int64


# 2. Business Time Period & Order Status Analysis

This section establishes the time coverage of the marketplace and examines
order-status distribution.

The analysis also identifies incomplete periods and investigates whether
missing operational dates are associated with specific order statuses.

This helps define the reliable time window used for monthly trend analysis.

In [10]:
# STEP 4 — BUSINESS TIME PERIOD ANALYSIS

orders = data["orders"]

print("STEP 4 — BUSINESS TIME PERIOD ANALYSIS")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    print(f"\n{col}")
    print(f"Earliest: {orders[col].min()}")
    print(f"Latest  : {orders[col].max()}")
    print(f"Missing : {orders[col].isna().sum()}")

STEP 4 — BUSINESS TIME PERIOD ANALYSIS

order_purchase_timestamp
Earliest: 2016-09-04 21:15:19
Latest  : 2018-10-17 17:30:18
Missing : 0

order_approved_at
Earliest: 2016-09-15 12:16:38
Latest  : 2018-09-03 17:40:06
Missing : 160

order_delivered_carrier_date
Earliest: 2016-10-08 10:34:01
Latest  : 2018-09-11 19:48:28
Missing : 1783

order_delivered_customer_date
Earliest: 2016-10-11 13:46:32
Latest  : 2018-10-17 13:22:46
Missing : 2965

order_estimated_delivery_date
Earliest: 2016-09-30
Latest  : 2018-11-12
Missing : 0


In [11]:
# STEP 4B — MISSING DATES VS ORDER STATUS

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for col in date_columns:
    print("\n" + "=" * 70)
    print(f"MISSING {col.upper()} BY ORDER STATUS")
    print("=" * 70)

    missing_status = (
        orders[orders[col].isna()]
        .groupby("order_status")
        .size()
        .sort_values(ascending=False)
    )

    print(missing_status)
    print(f"\nTotal missing: {orders[col].isna().sum()}")


MISSING ORDER_APPROVED_AT BY ORDER STATUS
order_status
canceled     141
delivered     14
created        5
dtype: int64

Total missing: 160

MISSING ORDER_DELIVERED_CARRIER_DATE BY ORDER STATUS
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
dtype: int64

Total missing: 1783

MISSING ORDER_DELIVERED_CUSTOMER_DATE BY ORDER STATUS
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
dtype: int64

Total missing: 2965


In [12]:
# STEP 5 — ORDER STATUS ANALYSIS

print("STEP 5 — ORDER STATUS ANALYSIS")

status_summary = (
    orders["order_status"]
    .value_counts()
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

status_summary["percentage"] = (
    status_summary["order_count"] / len(orders) * 100
).round(2)

print(status_summary)

print("\nTotal orders:", len(orders))

STEP 5 — ORDER STATUS ANALYSIS
  order_status  order_count  percentage
0    delivered        96478       97.02
1      shipped         1107        1.11
2     canceled          625        0.63
3  unavailable          609        0.61
4     invoiced          314        0.32
5   processing          301        0.30
6      created            5        0.01
7     approved            2        0.00

Total orders: 99441


In [13]:
# STEP 5B — ORDER VALUE BY STATUS

order_items = data["order_items"]

# Calculate total product and freight value per order
order_value = (
    order_items
    .groupby("order_id")
    .agg(
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_value["total_order_value"] = (
    order_value["product_value"] +
    order_value["freight_value"]
)

# Merge with order status
status_value_analysis = (
    orders[["order_id", "order_status"]]
    .merge(order_value, on="order_id", how="left")
    .groupby("order_status")
    .agg(
        total_orders=("order_id", "count"),
        orders_with_items=("total_order_value", "count"),
        total_product_value=("product_value", "sum"),
        total_freight_value=("freight_value", "sum"),
        total_order_value=("total_order_value", "sum")
    )
    .sort_values("total_order_value", ascending=False)
)

print("STEP 5B — ORDER VALUE BY STATUS")

print(status_value_analysis.round(2))

STEP 5B — ORDER VALUE BY STATUS
              total_orders  orders_with_items  total_product_value  \
order_status                                                         
delivered            96478              96478          13221498.11   
shipped               1107               1106            150727.44   
canceled               625                461             95235.27   
processing             301                301             60439.22   
invoiced               314                312             61526.37   
unavailable            609                  6              2007.69   
approved                 2                  2               209.60   
created                  5                  0                 0.00   

              total_freight_value  total_order_value  
order_status                                          
delivered              2198275.64        15419773.75  
shipped                  26401.90          177129.34  
canceled                 10650.45          1058

# 3. Sales & Order Trend Analysis

The objective of this section is to understand how order volume and delivered
sales changed over time.

Key metrics include:

- Monthly order volume
- Monthly delivered sales
- Month-over-month growth
- Average Order Value (AOV)

Because the dataset contains partial months at the beginning and end of the
observation period, complete-month analysis is used when comparing business
trends.

In [14]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

print(orders["order_purchase_timestamp"].dtype)
print("Missing:", orders["order_purchase_timestamp"].isna().sum())

datetime64[us]
Missing: 0


In [15]:
# STEP 6 — MONTHLY ORDER TREND

monthly_orders = (
    orders
    .assign(
        order_month=orders["order_purchase_timestamp"].dt.to_period("M")
    )
    .groupby("order_month")
    .agg(
        total_orders=("order_id", "count")
    )
)

print("STEP 6 — MONTHLY ORDER TREND")

print(monthly_orders)

print("\nNumber of months:", len(monthly_orders))
print("Highest order month:")
print(monthly_orders.loc[monthly_orders["total_orders"].idxmax()])

print("\nLowest order month:")
print(monthly_orders.loc[monthly_orders["total_orders"].idxmin()])

STEP 6 — MONTHLY ORDER TREND
             total_orders
order_month              
2016-09                 4
2016-10               324
2016-12                 1
2017-01               800
2017-02              1780
2017-03              2682
2017-04              2404
2017-05              3700
2017-06              3245
2017-07              4026
2017-08              4331
2017-09              4285
2017-10              4631
2017-11              7544
2017-12              5673
2018-01              7269
2018-02              6728
2018-03              7211
2018-04              6939
2018-05              6873
2018-06              6167
2018-07              6292
2018-08              6512
2018-09                16
2018-10                 4

Number of months: 25
Highest order month:
total_orders    7544
Name: 2017-11, dtype: int64

Lowest order month:
total_orders    1
Name: 2016-12, dtype: int64


In [16]:
# STEP 6B — IDENTIFY COMPLETE MONTHS

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M")

monthly_orders = (
    orders
    .groupby("order_month")
    .size()
    .rename("total_orders")
    .reset_index()
)

monthly_orders["year"] = monthly_orders["order_month"].dt.year
monthly_orders["month"] = monthly_orders["order_month"].dt.month

print("MONTHLY ORDER COVERAGE")

print(monthly_orders.to_string(index=False))

MONTHLY ORDER COVERAGE
order_month  total_orders  year  month
    2016-09             4  2016      9
    2016-10           324  2016     10
    2016-12             1  2016     12
    2017-01           800  2017      1
    2017-02          1780  2017      2
    2017-03          2682  2017      3
    2017-04          2404  2017      4
    2017-05          3700  2017      5
    2017-06          3245  2017      6
    2017-07          4026  2017      7
    2017-08          4331  2017      8
    2017-09          4285  2017      9
    2017-10          4631  2017     10
    2017-11          7544  2017     11
    2017-12          5673  2017     12
    2018-01          7269  2018      1
    2018-02          6728  2018      2
    2018-03          7211  2018      3
    2018-04          6939  2018      4
    2018-05          6873  2018      5
    2018-06          6167  2018      6
    2018-07          6292  2018      7
    2018-08          6512  2018      8
    2018-09            16  2018      9
  

In [17]:
# Identify first and last order dates in each month

monthly_coverage = (
    orders
    .groupby("order_month")["order_purchase_timestamp"]
    .agg(
        first_order="min",
        last_order="max"
    )
    .reset_index()
)

monthly_coverage["days_covered"] = (
    monthly_coverage["last_order"].dt.normalize()
    - monthly_coverage["first_order"].dt.normalize()
).dt.days + 1

print("MONTH COVERAGE")

print(monthly_coverage.to_string(index=False))

MONTH COVERAGE
order_month         first_order          last_order  days_covered
    2016-09 2016-09-04 21:15:19 2016-09-15 12:16:38            12
    2016-10 2016-10-02 22:07:52 2016-10-22 08:25:27            21
    2016-12 2016-12-23 23:16:47 2016-12-23 23:16:47             1
    2017-01 2017-01-05 11:56:06 2017-01-31 23:37:58            27
    2017-02 2017-02-01 00:04:17 2017-02-28 23:47:08            28
    2017-03 2017-03-01 00:01:30 2017-03-31 23:54:45            31
    2017-04 2017-04-01 00:54:10 2017-04-30 23:48:13            30
    2017-05 2017-05-01 01:18:22 2017-05-31 22:59:52            31
    2017-06 2017-06-01 00:05:38 2017-06-30 23:20:08            30
    2017-07 2017-07-01 00:04:15 2017-07-31 23:55:27            31
    2017-08 2017-08-01 00:02:01 2017-08-31 23:55:57            31
    2017-09 2017-09-01 00:03:50 2017-09-30 23:59:15            30
    2017-10 2017-10-01 00:03:33 2017-10-31 23:50:58            31
    2017-11 2017-11-01 00:12:34 2017-11-30 23:36:03          

In [18]:
# STEP 6C — COMPLETE MONTH BUSINESS TREND

orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M")

complete_months = orders[
    (orders["order_month"] >= "2017-02") &
    (orders["order_month"] <= "2018-07")
]

monthly_trend = (
    complete_months
    .groupby("order_month")
    .agg(
        total_orders=("order_id", "count")
    )
    .reset_index()
)

monthly_trend["mom_growth_%"] = (
    monthly_trend["total_orders"]
    .pct_change() * 100
).round(2)

print("STEP 6C — COMPLETE-MONTH ORDER TREND")

print(monthly_trend.to_string(index=False))

print("\nHighest-order month:")
print(
    monthly_trend.loc[
        monthly_trend["total_orders"].idxmax()
    ]
)

print("\nLowest-order month:")
print(
    monthly_trend.loc[
        monthly_trend["total_orders"].idxmin()
    ]
)

STEP 6C — COMPLETE-MONTH ORDER TREND
order_month  total_orders  mom_growth_%
    2017-02          1780           NaN
    2017-03          2682         50.67
    2017-04          2404        -10.37
    2017-05          3700         53.91
    2017-06          3245        -12.30
    2017-07          4026         24.07
    2017-08          4331          7.58
    2017-09          4285         -1.06
    2017-10          4631          8.07
    2017-11          7544         62.90
    2017-12          5673        -24.80
    2018-01          7269         28.13
    2018-02          6728         -7.44
    2018-03          7211          7.18
    2018-04          6939         -3.77
    2018-05          6873         -0.95
    2018-06          6167        -10.27
    2018-07          6292          2.03

Highest-order month:
order_month     2017-11
total_orders       7544
mom_growth_%       62.9
Name: 9, dtype: object

Lowest-order month:
order_month     2017-02
total_orders       1780
mom_growth_%     

In [19]:
# STEP 6D — MONTHLY DELIVERED SALES TREND

# Delivered orders only
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

# Aggregate order-item value at order level
order_sales = (
    data["order_items"]
    .groupby("order_id")
    .agg(
        product_value=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_sales["sales_value"] = (
    order_sales["product_value"] +
    order_sales["freight_value"]
)

# Merge sales value with delivered orders
delivered_sales = delivered_orders[
    ["order_id", "order_purchase_timestamp"]
].merge(
    order_sales[["order_id", "product_value", "freight_value", "sales_value"]],
    on="order_id",
    how="left"
)

delivered_sales["order_month"] = (
    delivered_sales["order_purchase_timestamp"]
    .dt.to_period("M")
)

monthly_sales = (
    delivered_sales
    .groupby("order_month")
    .agg(
        delivered_orders=("order_id", "count"),
        product_sales=("product_value", "sum"),
        freight_value=("freight_value", "sum"),
        total_sales_value=("sales_value", "sum")
    )
    .reset_index()
)

# Keep the same complete comparison period
monthly_sales = monthly_sales[
    (monthly_sales["order_month"] >= "2017-02") &
    (monthly_sales["order_month"] <= "2018-07")
].copy()

monthly_sales["mom_sales_growth_%"] = (
    monthly_sales["total_sales_value"]
    .pct_change() * 100
).round(2)

print("STEP 6D — MONTHLY DELIVERED SALES TREND")

print(monthly_sales.to_string(index=False))

print("\nHighest sales month:")
print(
    monthly_sales.loc[
        monthly_sales["total_sales_value"].idxmax()
    ]
)

STEP 6D — MONTHLY DELIVERED SALES TREND
order_month  delivered_orders  product_sales  freight_value  total_sales_value  mom_sales_growth_%
    2017-02              1653      234223.40       37015.92          271239.32                 NaN
    2017-03              2546      359198.85       55132.10          414330.95               52.75
    2017-04              2303      340669.68       50142.72          390812.40               -5.68
    2017-05              3546      489338.25       77513.15          566851.40               45.04
    2017-06              3135      421923.37       68127.00          490050.37              -13.55
    2017-07              3872      481604.52       84694.56          566299.08               15.56
    2017-08              4193      554699.70       91132.66          645832.36               14.04
    2017-09              4150      607399.67       93677.82          701077.49                8.55
    2017-10              4478      648247.65      102869.36          

In [20]:
# STEP 6E — MONTHLY AVERAGE ORDER VALUE

monthly_sales["aov"] = (
    monthly_sales["total_sales_value"]
    / monthly_sales["delivered_orders"]
).round(2)

monthly_sales["aov_growth_%"] = (
    monthly_sales["aov"].pct_change() * 100
).round(2)

print("STEP 6E — MONTHLY AVERAGE ORDER VALUE")

print(
    monthly_sales[
        [
            "order_month",
            "delivered_orders",
            "total_sales_value",
            "aov",
            "aov_growth_%"
        ]
    ].to_string(index=False)
)

print("\nHighest AOV month:")
print(
    monthly_sales.loc[
        monthly_sales["aov"].idxmax()
    ]
)

print("\nLowest AOV month:")
print(
    monthly_sales.loc[
        monthly_sales["aov"].idxmin()
    ]
)

STEP 6E — MONTHLY AVERAGE ORDER VALUE
order_month  delivered_orders  total_sales_value    aov  aov_growth_%
    2017-02              1653          271239.32 164.09           NaN
    2017-03              2546          414330.95 162.74         -0.82
    2017-04              2303          390812.40 169.70          4.28
    2017-05              3546          566851.40 159.86         -5.80
    2017-06              3135          490050.37 156.32         -2.21
    2017-07              3872          566299.08 146.25         -6.44
    2017-08              4193          645832.36 154.03          5.32
    2017-09              4150          701077.49 168.93          9.67
    2017-10              4478          751117.01 167.73         -0.71
    2017-11              7289         1153364.20 158.23         -5.66
    2017-12              5513          843078.29 152.93         -3.35
    2018-01              7069         1077887.46 152.48         -0.29
    2018-02              6555          966168.41 147

# 4. Product & Category Performance

This section evaluates product and category performance using delivered
orders.

Product categories are mapped to an analysis-ready category label using the
product and category-translation tables.

The analysis focuses on:

- Category sales
- Item volume
- Average item price
- Top-performing products
- Category-level customer satisfaction

The analysis avoids directly joining multiple one-to-many tables at the
order level to prevent row duplication.

In [21]:
# STEP 7 — PRODUCT / CATEGORY MAPPING VALIDATION

items = data["order_items"]
products = data["products"]
translation = data["product_category_name_translation"]

print("STEP 7 — PRODUCT / CATEGORY MAPPING VALIDATION")

# Product IDs used in order items but missing from products
missing_products = (
    set(items["product_id"].dropna())
    - set(products["product_id"].dropna())
)

print("\nProducts in order_items but missing from products:")
print(len(missing_products))

# Categories in products but missing from translation
product_categories = set(
    products["product_category_name"].dropna()
)

translated_categories = set(
    translation["product_category_name"].dropna()
)

missing_translations = product_categories - translated_categories

print("\nProduct categories without English translation:")
print(len(missing_translations))

if missing_translations:
    print(sorted(missing_translations))

# Products without category
products_without_category = products["product_category_name"].isna().sum()

print("\nProducts without category:")
print(products_without_category)

STEP 7 — PRODUCT / CATEGORY MAPPING VALIDATION

Products in order_items but missing from products:
0

Product categories without English translation:
2
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

Products without category:
610


In [22]:
# ============================================================
# STEP 7B — BUILD PRODUCT-CATEGORY ANALYSIS TABLE
# ============================================================

items = data["order_items"]
products = data["products"]
translation = data["product_category_name_translation"]

product_category = (
    products[
        ["product_id", "product_category_name"]
    ]
    .merge(
        translation[
            ["product_category_name", "product_category_name_english"]
        ],
        on="product_category_name",
        how="left"
    )
)

# Create a stable analytical category label
product_category["analysis_category"] = (
    product_category["product_category_name_english"]
    .fillna(product_category["product_category_name"])
    .fillna("uncategorized")
)

print("STEP 7B — PRODUCT-CATEGORY ANALYSIS TABLE")

print("\nRows:", len(product_category))
print("Unique products:", product_category["product_id"].nunique())

print("\nCategory label missing:")
print(product_category["analysis_category"].isna().sum())

print("\nUntranslated/original categories being used as fallback:")
fallback = product_category[
    product_category["product_category_name_english"].isna()
    & product_category["product_category_name"].notna()
]

print(fallback["product_category_name"].value_counts())

STEP 7B — PRODUCT-CATEGORY ANALYSIS TABLE

Rows: 32951
Unique products: 32951

Category label missing:
0

Untranslated/original categories being used as fallback:
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


In [23]:
# ============================================================
# STEP 7C — ATTACH PRODUCT CATEGORY TO ORDER ITEMS
# ============================================================

items = data["order_items"]

item_analysis = items.merge(
    product_category[
        [
            "product_id",
            "analysis_category"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("STEP 7C — ORDER ITEMS + PRODUCT CATEGORY")

print("\nRows:", len(item_analysis))
print("Expected rows:", len(items))

print("\nUnique order IDs:", item_analysis["order_id"].nunique())
print("Unique products:", item_analysis["product_id"].nunique())

print("\nMissing category after merge:")
print(item_analysis["analysis_category"].isna().sum())

print("\nCategory distribution:")
print(
    item_analysis["analysis_category"]
    .value_counts()
    .head(20)
)

STEP 7C — ORDER ITEMS + PRODUCT CATEGORY

Rows: 112650
Expected rows: 112650

Unique order IDs: 98666
Unique products: 32951

Missing category after merge:
0

Category distribution:
analysis_category
bed_bath_table              11115
health_beauty                9670
sports_leisure               8641
furniture_decor              8334
computers_accessories        7827
housewares                   6964
watches_gifts                5991
telephony                    4545
garden_tools                 4347
auto                         4235
toys                         4117
cool_stuff                   3796
perfumery                    3419
baby                         3065
electronics                  2767
stationery                   2517
fashion_bags_accessories     2031
pet_shop                     1947
office_furniture             1691
uncategorized                1603
Name: count, dtype: int64


In [24]:
# ============================================================
# STEP 7D — CATEGORY PERFORMANCE ANALYSIS
# ============================================================

category_performance = (
    item_analysis
    .groupby("analysis_category")
    .agg(
        item_count=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        product_sales=("price", "sum"),
        freight_value=("freight_value", "sum"),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

category_performance["total_sales_value"] = (
    category_performance["product_sales"]
    + category_performance["freight_value"]
)

category_performance["sales_rank"] = (
    category_performance["total_sales_value"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

category_performance = category_performance.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 7D — CATEGORY PERFORMANCE")

display(category_performance.head(20))

STEP 7D — CATEGORY PERFORMANCE


,analysis_category,item_count,unique_orders,unique_products,unique_sellers,product_sales,freight_value,average_item_price,total_sales_value,sales_rank
43,health_beauty,9670,8836,2444,492,1258681.34,182566.73,130.163531,1441248.07,1
73,watches_gifts,5991,5624,1329,101,1205005.68,100535.93,201.135984,1305541.61,2
7,bed_bath_table,11115,9417,3029,196,1036988.68,204693.04,93.296327,1241681.72,3
67,sports_leisure,8641,7720,2867,481,988048.97,168607.51,114.344285,1156656.48,4
15,computers_accessories,7827,6689,1639,287,911954.32,147318.08,116.513903,1059272.40,5
39,furniture_decor,8334,6449,2657,370,729762.49,172749.30,87.564494,902511.79,6
49,housewares,6964,5884,2335,468,632248.66,146149.11,90.788148,778397.77,7
20,cool_stuff,3796,3632,789,267,635290.85,84039.10,167.357969,719329.95,8
5,auto,4235,3897,1900,383,592720.11,92664.21,139.957523,685384.32,9
42,garden_tools,4347,3518,753,237,485256.46,98962.75,111.630196,584219.21,10


In [25]:
# ============================================================
# STEP 7E — DELIVERED CATEGORY PERFORMANCE
# ============================================================

# Add order status to item-level data
item_analysis_status = item_analysis.merge(
    data["orders"][["order_id", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one"
)

# Keep delivered orders only
delivered_items = item_analysis_status[
    item_analysis_status["order_status"] == "delivered"
].copy()

category_delivered = (
    delivered_items
    .groupby("analysis_category")
    .agg(
        item_count=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        product_sales=("price", "sum"),
        freight_value=("freight_value", "sum"),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

category_delivered["total_sales_value"] = (
    category_delivered["product_sales"]
    + category_delivered["freight_value"]
)

category_delivered["sales_rank"] = (
    category_delivered["total_sales_value"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

category_delivered = category_delivered.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 7E — DELIVERED CATEGORY PERFORMANCE")

display(category_delivered.head(20))

STEP 7E — DELIVERED CATEGORY PERFORMANCE


,analysis_category,item_count,unique_orders,unique_products,unique_sellers,product_sales,freight_value,average_item_price,total_sales_value,sales_rank
43,health_beauty,9465,8647,2397,479,1233131.72,178957.81,130.283330,1412089.53,1
73,watches_gifts,5859,5495,1300,95,1166176.98,98156.14,199.040276,1264333.12,2
7,bed_bath_table,10953,9272,2991,189,1023434.76,201774.50,93.438762,1225209.26,3
67,sports_leisure,8431,7530,2822,466,954852.55,163404.36,113.254958,1118256.91,4
15,computers_accessories,7644,6530,1600,279,888724.61,143999.16,116.264339,1032723.77,5
39,furniture_decor,8160,6307,2593,352,711927.69,168402.23,87.246040,880329.92,6
49,housewares,6795,5743,2282,452,615628.69,142763.56,90.600249,758392.25,7
20,cool_stuff,3718,3559,770,259,610204.10,81476.79,164.121598,691680.89,8
5,auto,4140,3810,1853,371,578966.65,90488.10,139.847017,669454.75,9
42,garden_tools,4268,3448,725,227,470495.28,96650.40,110.237882,567145.68,10


In [26]:
# ============================================================
# STEP 8 — TOP PRODUCTS BY DELIVERED SALES
# ============================================================

product_performance = (
    delivered_items
    .groupby("product_id")
    .agg(
        item_count=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        product_sales=("price", "sum"),
        freight_value=("freight_value", "sum"),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

product_performance["total_sales_value"] = (
    product_performance["product_sales"]
    + product_performance["freight_value"]
)

product_performance = product_performance.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 8 — TOP PRODUCTS BY DELIVERED SALES")

display(product_performance.head(20))

STEP 8 — TOP PRODUCTS BY DELIVERED SALES


,product_id,item_count,unique_orders,unique_sellers,product_sales,freight_value,average_item_price,total_sales_value
23546,bb50f2e236e5eea0100680137654686c,194,186,1,63560.00,3698.03,327.628866,67258.03
26436,d1c427060a0f73f6b889a5c7c61f2ac4,332,313,1,45620.56,13336.75,137.411325,58957.31
13749,6cdd53843498f92890544667809f1595,153,148,1,53652.30,4281.43,350.668627,57933.73
19290,99a4788cb24856965c36a24e339b6058,477,456,2,42049.66,7857.84,88.154423,49907.50
7881,3dd2a17168ec895c781a9191c1e95ad7,272,253,1,40782.80,7093.26,149.936765,47876.06
26997,d6160fb7873f184099d9bc95e30376af,33,33,1,45949.35,1364.83,1392.404545,47314.18
21617,aca2eb7d00ea1a7b8ebd4e68314663af,520,425,1,37104.30,7093.45,71.354423,44197.75
12074,5f504b3a1c75b73d6151be81eb05bdc9,63,63,1,37733.90,3991.91,598.950794,41725.81
4892,25c38557cf793876c5abdd5931f922db,38,38,2,38907.32,1404.63,1023.876842,40311.95
10616,53b36df67ebb7c41585e8d54d6772e08,321,304,3,37454.63,2258.86,116.681090,39713.49


In [27]:
# Add category to product performance

product_performance = product_performance.merge(
    product_category[
        ["product_id", "analysis_category"]
    ],
    on="product_id",
    how="left",
    validate="one_to_one"
)

display(
    product_performance[
        [
            "product_id",
            "analysis_category",
            "item_count",
            "unique_orders",
            "product_sales",
            "freight_value",
            "total_sales_value",
            "average_item_price"
        ]
    ].head(20)
)

,product_id,analysis_category,item_count,unique_orders,product_sales,freight_value,total_sales_value,average_item_price
0,bb50f2e236e5eea0100680137654686c,health_beauty,194,186,63560.00,3698.03,67258.03,327.628866
1,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,332,313,45620.56,13336.75,58957.31,137.411325
2,6cdd53843498f92890544667809f1595,health_beauty,153,148,53652.30,4281.43,57933.73,350.668627
3,99a4788cb24856965c36a24e339b6058,bed_bath_table,477,456,42049.66,7857.84,49907.50,88.154423
4,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,272,253,40782.80,7093.26,47876.06,149.936765
5,d6160fb7873f184099d9bc95e30376af,computers,33,33,45949.35,1364.83,47314.18,1392.404545
6,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,425,37104.30,7093.45,44197.75,71.354423
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,63,37733.90,3991.91,41725.81,598.950794
8,25c38557cf793876c5abdd5931f922db,baby,38,38,38907.32,1404.63,40311.95,1023.876842
9,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,321,304,37454.63,2258.86,39713.49,116.681090


In [28]:
# ============================================================
# STEP 8B — BEST-SELLING PRODUCTS BY VOLUME
# ============================================================

product_volume = (
    delivered_items
    .groupby("product_id")
    .agg(
        items_sold=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        total_sales_value=("price", "sum")
    )
    .reset_index()
)

product_volume = product_volume.merge(
    product_category[
        ["product_id", "analysis_category"]
    ],
    on="product_id",
    how="left",
    validate="one_to_one"
)

product_volume = product_volume.sort_values(
    ["items_sold", "total_sales_value"],
    ascending=[False, False]
)

print("STEP 8B — BEST-SELLING PRODUCTS BY ITEMS SOLD")

display(
    product_volume[
        [
            "product_id",
            "analysis_category",
            "items_sold",
            "unique_orders",
            "total_sales_value"
        ]
    ].head(20)
)

STEP 8B — BEST-SELLING PRODUCTS BY ITEMS SOLD


,product_id,analysis_category,items_sold,unique_orders,total_sales_value
21617,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,520,425,37104.30
8429,422879e10f46682990de24d770e7f83d,garden_tools,484,352,26577.22
19290,99a4788cb24856965c36a24e339b6058,bed_bath_table,477,456,42049.66
7206,389d119b48cf3043d311335e499d9c6b,garden_tools,390,309,21336.79
6926,368c6c730842d78016ad823897a372db,garden_tools,388,291,21056.80
10589,53759a2ecddad2bb87a079a1f1519f73,garden_tools,373,287,20387.20
26436,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,332,313,45620.56
10616,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,321,304,37454.63
2734,154e7e31ebfa092203795c972e5804a6,health_beauty,274,262,6173.26
7881,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,272,253,40782.80


In [29]:
# ============================================================
# STEP 8C — LOW-SELLING PRODUCTS
# ============================================================

low_selling_products = product_volume[
    product_volume["items_sold"] <= 2
].copy()

print("STEP 8C — LOW-SELLING PRODUCTS")

print("Products with 1–2 delivered item sales:",
      len(low_selling_products))

print("\nDistribution:")
print(
    low_selling_products["items_sold"]
    .value_counts()
    .sort_index()
)

display(
    low_selling_products[
        [
            "product_id",
            "analysis_category",
            "items_sold",
            "unique_orders",
            "total_sales_value"
        ]
    ].sort_values(
        ["items_sold", "total_sales_value"],
        ascending=[True, False]
    ).head(20)
)

STEP 8C — LOW-SELLING PRODUCTS
Products with 1–2 delivered item sales: 23385

Distribution:
items_sold
1    17716
2     5669
Name: count, dtype: int64


,product_id,analysis_category,items_sold,unique_orders,total_sales_value
9227,489ae2aa008f021502940f251d4cce7f,housewares,1,1,6735.00
13382,69c590f7ffc7bf8db97190b6cb6ed62e,computers,1,1,6729.00
3620,1bdf5e6731585cf01aa8169c7028d6ad,art,1,1,6499.00
20835,a6492cc69376c469ab6f61d8f44de961,small_appliances,1,1,4799.00
24639,c3ed642d592594bb648ff4a04cee2747,small_appliances,1,1,4690.00
4863,259037a6a41845e455183f89c5035f18,computers,1,1,4590.00
20308,a1beef8f3992dbd4cd8726796aa69c53,musical_instruments,1,1,4399.87
13751,6cdf8fc1d741c76586d8b6b15e9eef30,consoles_games,1,1,4099.99
13282,6902c1962dd19d540807d0ab8fade5c6,watches_gifts,1,1,3999.90
9742,4ca7b91a31637bd24fb8e559d5e015e4,small_appliances,1,1,3999.00


# 5. Seller Performance & Geographic Distribution

This section evaluates seller contribution and geographic concentration.

The analysis covers:

- Top sellers by delivered sales
- Seller-state sales contribution
- Sales per seller
- Sales per order
- Sales per item

Minimum-volume thresholds are applied where necessary to avoid interpreting
very small seller populations as reliable benchmarks.

In [30]:
# ============================================================
# STEP 9 — SELLER PERFORMANCE
# ============================================================

seller_performance = (
    delivered_items
    .groupby("seller_id")
    .agg(
        items_sold=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        unique_products=("product_id", "nunique"),
        product_sales=("price", "sum"),
        freight_value=("freight_value", "sum"),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

seller_performance["total_sales_value"] = (
    seller_performance["product_sales"]
    + seller_performance["freight_value"]
)

seller_performance = seller_performance.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 9 — TOP SELLERS BY DELIVERED SALES")

display(seller_performance.head(20))

STEP 9 — TOP SELLERS BY DELIVERED SALES


,seller_id,items_sold,unique_orders,unique_products,product_sales,freight_value,average_item_price,total_sales_value
834,4869f7a5dfa277a7dca6462dcf3b52b2,1148,1124,94,226987.93,20019.13,197.724678,247007.06
1480,7c67e1448b00f6e969d365cea6b010ab,1355,973,197,186570.05,51236.64,137.690074,237806.69
858,4a3ca9315b744ce9f8e9374361493884,1949,1772,394,196882.12,34338.31,101.016993,231220.43
982,53243585a1d6dc2643021fd1853d8905,400,348,23,217940.44,12856.58,544.851100,230797.02
2903,fa1c13f2614d7b5c4749cbc52fecda94,579,578,284,190917.14,9916.36,329.735993,200833.50
2543,da8622b14eb17ae2831f4ac5b9dab84a,1548,1311,221,159816.87,24889.91,103.240872,184706.78
1504,7e93a43ef30c4f03f38b393420bc753a,322,319,175,165981.49,5992.06,515.470466,171973.55
188,1025f0e2d44d7041d6cf58b6550e0bfa,1420,910,153,138208.56,33716.40,97.329972,171924.96
1450,7a67c85e85bb2ce8582c35f2203ad736,1155,1145,146,139658.69,20619.83,120.916615,160278.52
1758,955fee9216a65b617aa5c0531780ce60,1472,1261,103,131836.71,24769.77,89.562982,156606.48


In [31]:
# Add seller location

seller_performance = seller_performance.merge(
    data["sellers"][
        ["seller_id", "seller_city", "seller_state"]
    ],
    on="seller_id",
    how="left",
    validate="one_to_one"
)

display(
    seller_performance[
        [
            "seller_id",
            "seller_city",
            "seller_state",
            "items_sold",
            "unique_orders",
            "unique_products",
            "product_sales",
            "freight_value",
            "total_sales_value",
            "average_item_price"
        ]
    ].head(20)
)

,seller_id,seller_city,seller_state,items_sold,unique_orders,unique_products,product_sales,freight_value,total_sales_value,average_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1148,1124,94,226987.93,20019.13,247007.06,197.724678
1,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1355,973,197,186570.05,51236.64,237806.69,137.690074
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1949,1772,394,196882.12,34338.31,231220.43,101.016993
3,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,400,348,23,217940.44,12856.58,230797.02,544.851100
4,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,579,578,284,190917.14,9916.36,200833.50,329.735993
5,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1548,1311,221,159816.87,24889.91,184706.78,103.240872
6,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,322,319,175,165981.49,5992.06,171973.55,515.470466
7,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1420,910,153,138208.56,33716.40,171924.96,97.329972
8,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1155,1145,146,139658.69,20619.83,160278.52,120.916615
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1472,1261,103,131836.71,24769.77,156606.48,89.562982


In [32]:
# ============================================================
# STEP 9B — SELLER STATE PERFORMANCE
# ============================================================

seller_state_performance = (
    seller_performance
    .groupby("seller_state")
    .agg(
        seller_count=("seller_id", "nunique"),
        items_sold=("items_sold", "sum"),
        unique_orders=("unique_orders", "sum"),
        unique_products=("unique_products", "sum"),
        product_sales=("product_sales", "sum"),
        freight_value=("freight_value", "sum"),
        total_sales_value=("total_sales_value", "sum")
    )
    .reset_index()
    .sort_values("total_sales_value", ascending=False)
)

print("STEP 9B — SELLER STATE PERFORMANCE")

display(seller_state_performance)

STEP 9B — SELLER STATE PERFORMANCE


,seller_state,seller_count,items_sold,unique_orders,unique_products,product_sales,freight_value,total_sales_value
21,SP,1769,78604,69401,23147,8509511.46,1447545.45,9957056.91
14,PR,335,8487,7556,3002,1232096.99,192064.26,1424161.25
7,MG,236,8603,7747,2704,977866.31,206561.23,1184427.54
15,RJ,163,4685,4229,1483,820611.59,91307.08,911918.67
19,SC,184,4000,3608,1428,613591.65,104147.72,717739.37
18,RS,125,2169,1964,771,373412.08,56337.01,429749.09
1,BA,18,624,550,122,277925.51,19221.42,297146.93
3,DF,30,883,808,374,94840.31,18052.47,112892.78
12,PE,9,445,403,63,91164.15,12321.95,103486.10
5,GO,39,508,451,234,64806.59,12282.89,77089.48


In [33]:
# Seller sales concentration

total_seller_sales = seller_state_performance["total_sales_value"].sum()

seller_state_performance["sales_share_%"] = (
    seller_state_performance["total_sales_value"]
    / total_seller_sales
    * 100
).round(2)

display(
    seller_state_performance[
        [
            "seller_state",
            "seller_count",
            "items_sold",
            "unique_orders",
            "total_sales_value",
            "sales_share_%"
        ]
    ]
)

,seller_state,seller_count,items_sold,unique_orders,total_sales_value,sales_share_%
21,SP,1769,78604,69401,9957056.91,64.57
14,PR,335,8487,7556,1424161.25,9.24
7,MG,236,8603,7747,1184427.54,7.68
15,RJ,163,4685,4229,911918.67,5.91
19,SC,184,4000,3608,717739.37,4.65
18,RS,125,2169,1964,429749.09,2.79
1,BA,18,624,550,297146.93,1.93
3,DF,30,883,808,112892.78,0.73
12,PE,9,445,403,103486.10,0.67
5,GO,39,508,451,77089.48,0.50


In [34]:
# ============================================================
# STEP 9C — SELLER STATE PRODUCTIVITY
# ============================================================

seller_state_productivity = seller_state_performance.copy()

seller_state_productivity["sales_per_seller"] = (
    seller_state_productivity["total_sales_value"]
    / seller_state_productivity["seller_count"]
)

seller_state_productivity["sales_per_order"] = (
    seller_state_productivity["total_sales_value"]
    / seller_state_productivity["unique_orders"]
)

seller_state_productivity["sales_per_item"] = (
    seller_state_productivity["total_sales_value"]
    / seller_state_productivity["items_sold"]
)

seller_state_productivity = seller_state_productivity.sort_values(
    "sales_per_seller",
    ascending=False
)

print("STEP 9C — SELLER STATE PRODUCTIVITY")

display(
    seller_state_productivity[
        [
            "seller_state",
            "seller_count",
            "unique_orders",
            "items_sold",
            "total_sales_value",
            "sales_share_%",
            "sales_per_seller",
            "sales_per_order",
            "sales_per_item"
        ]
    ]
)

STEP 9C — SELLER STATE PRODUCTIVITY


,seller_state,seller_count,unique_orders,items_sold,total_sales_value,sales_share_%,sales_per_seller,sales_per_order,sales_per_item
6,MA,1,389,402,48169.50,0.31,48169.500000,123.829049,119.824627
1,BA,18,550,624,297146.93,1.93,16508.162778,540.267145,476.197003
12,PE,9,403,445,103486.10,0.67,11498.455556,256.789330,232.553034
21,SP,1769,69401,78604,9957056.91,64.57,5628.635902,143.471375,126.673667
15,RJ,163,4229,4685,911918.67,5.91,5594.593067,215.634587,194.646461
9,MT,4,136,144,21623.67,0.14,5405.917500,158.997574,150.164375
7,MG,236,7747,8603,1184427.54,7.68,5018.760763,152.888543,137.676106
14,PR,335,7556,8487,1424161.25,9.24,4251.227612,188.480843,167.805025
19,SC,184,3608,4000,717739.37,4.65,3900.757446,198.929981,179.434843
3,DF,30,808,883,112892.78,0.73,3763.092667,139.718787,127.851393


In [35]:
# ============================================================
# STEP 9D — SELLER STATE PRODUCTIVITY
# WITH MINIMUM SELLER COUNT
# ============================================================

# Keep states with at least 20 sellers
seller_state_reliable = seller_state_productivity[
    seller_state_productivity["seller_count"] >= 20
].copy()

seller_state_reliable = seller_state_reliable.sort_values(
    "sales_per_seller",
    ascending=False
)

print("STEP 9D — SELLER STATE PRODUCTIVITY")
print("Minimum seller count: 20")

display(
    seller_state_reliable[
        [
            "seller_state",
            "seller_count",
            "unique_orders",
            "items_sold",
            "total_sales_value",
            "sales_share_%",
            "sales_per_seller",
            "sales_per_order",
            "sales_per_item"
        ]
    ]
)

STEP 9D — SELLER STATE PRODUCTIVITY
Minimum seller count: 20


,seller_state,seller_count,unique_orders,items_sold,total_sales_value,sales_share_%,sales_per_seller,sales_per_order,sales_per_item
21,SP,1769,69401,78604,9957056.91,64.57,5628.635902,143.471375,126.673667
15,RJ,163,4229,4685,911918.67,5.91,5594.593067,215.634587,194.646461
7,MG,236,7747,8603,1184427.54,7.68,5018.760763,152.888543,137.676106
14,PR,335,7556,8487,1424161.25,9.24,4251.227612,188.480843,167.805025
19,SC,184,3608,4000,717739.37,4.65,3900.757446,198.929981,179.434843
3,DF,30,808,883,112892.78,0.73,3763.092667,139.718787,127.851393
18,RS,125,1964,2169,429749.09,2.79,3437.992720,218.813182,198.132361
4,ES,22,310,364,58853.49,0.38,2675.158636,189.849968,161.685412
5,GO,39,451,508,77089.48,0.50,1976.653333,170.930111,151.750945


# 6. Customer Analysis & Retention

This section analyzes customer geography, purchasing frequency, and repeat
behavior.

Key questions include:

- Where are customers concentrated?
- What proportion of customers make repeat purchases?
- How frequently do repeat customers purchase?
- How valuable are repeat customers compared with one-time customers?
- How does retention vary across customer cohorts?

Repeat-customer metrics are calculated using customers with at least one
delivered order so that the retention denominator is clearly defined.

In [36]:
# ============================================================
# STEP 10 — CUSTOMER STATE PERFORMANCE
# ============================================================

customers = data["customers"]

# Connect delivered orders to customer information
customer_orders = (
    orders[
        orders["order_status"] == "delivered"
    ][
        ["order_id", "customer_id"]
    ]
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id",
                "customer_city",
                "customer_state"
            ]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

# Attach delivered order value
customer_orders = customer_orders.merge(
    delivered_sales[
        ["order_id", "sales_value"]
    ],
    on="order_id",
    how="left",
    validate="one_to_one"
)

customer_state_performance = (
    customer_orders
    .groupby("customer_state")
    .agg(
        unique_customers=("customer_unique_id", "nunique"),
        delivered_orders=("order_id", "nunique"),
        total_sales_value=("sales_value", "sum")
    )
    .reset_index()
)

customer_state_performance["sales_share_%"] = (
    customer_state_performance["total_sales_value"]
    / customer_state_performance["total_sales_value"].sum()
    * 100
).round(2)

customer_state_performance = customer_state_performance.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 10 — CUSTOMER STATE PERFORMANCE")

display(customer_state_performance)

STEP 10 — CUSTOMER STATE PERFORMANCE


,customer_state,unique_customers,delivered_orders,total_sales_value,sales_share_%
25,SP,39156,40501,5769703.15,37.42
18,RJ,11917,12350,2055401.57,13.33
10,MG,11001,11354,1818891.67,11.80
22,RS,5168,5345,861472.79,5.59
17,PR,4769,4923,781708.80,5.07
23,SC,3449,3546,595127.78,3.86
4,BA,3158,3256,591137.81,3.83
6,DF,2019,2080,346123.35,2.24
8,GO,1895,1957,334212.35,2.17
7,ES,1928,1995,317657.93,2.06


In [37]:
# ============================================================
# STEP 10B — CUSTOMER STATE PRODUCTIVITY
# ============================================================

customer_state_productivity = customer_state_performance.copy()

customer_state_productivity["sales_per_customer"] = (
    customer_state_productivity["total_sales_value"]
    / customer_state_productivity["unique_customers"]
)

customer_state_productivity["sales_per_order"] = (
    customer_state_productivity["total_sales_value"]
    / customer_state_productivity["delivered_orders"]
)

customer_state_productivity["orders_per_customer"] = (
    customer_state_productivity["delivered_orders"]
    / customer_state_productivity["unique_customers"]
)

customer_state_productivity = customer_state_productivity.sort_values(
    "sales_per_customer",
    ascending=False
)

print("STEP 10B — CUSTOMER STATE PRODUCTIVITY")

display(
    customer_state_productivity[
        [
            "customer_state",
            "unique_customers",
            "delivered_orders",
            "total_sales_value",
            "sales_share_%",
            "sales_per_customer",
            "sales_per_order",
            "orders_per_customer"
        ]
    ]
)

STEP 10B — CUSTOMER STATE PRODUCTIVITY


,customer_state,unique_customers,delivered_orders,total_sales_value,sales_share_%,sales_per_customer,sales_per_order,orders_per_customer
14,PB,504,517,137838.55,0.89,273.489187,266.612282,1.025794
0,AC,76,80,19575.33,0.13,257.570132,244.691625,1.052632
20,RO,231,243,56966.00,0.37,246.606061,234.427984,1.051948
3,AP,66,67,16141.81,0.10,244.572879,240.922537,1.015152
1,AL,387,397,94172.49,0.61,243.339767,237.210302,1.025840
13,PA,922,946,212023.57,1.38,229.960488,224.126395,1.026030
16,PI,464,476,105178.19,0.68,226.677134,220.962584,1.025862
21,RR,40,41,9039.52,0.06,225.988000,220.476098,1.025000
26,TO,267,274,60007.37,0.39,224.746704,219.005000,1.026217
19,RN,464,474,100714.78,0.65,217.057716,212.478439,1.021552


In [38]:
# ============================================================
# STEP 10C — REPEAT CUSTOMER ANALYSIS
# ============================================================

customer_order_counts = (
    customer_orders
    .groupby("customer_unique_id")
    .agg(
        delivered_orders=("order_id", "nunique"),
        total_customer_sales=("sales_value", "sum")
    )
    .reset_index()
)

customer_order_counts["customer_type"] = (
    customer_order_counts["delivered_orders"]
    .apply(
        lambda x: "Repeat Customer" if x > 1 else "One-Time Customer"
    )
)

customer_summary = (
    customer_order_counts["customer_type"]
    .value_counts()
    .rename_axis("customer_type")
    .reset_index(name="customer_count")
)

customer_summary["customer_share_%"] = (
    customer_summary["customer_count"]
    / customer_summary["customer_count"].sum()
    * 100
).round(2)

print("STEP 10C — REPEAT CUSTOMER ANALYSIS")

print("\nCustomer summary:")
display(customer_summary)

print("\nOrder-count distribution:")
print(
    customer_order_counts["delivered_orders"]
    .value_counts()
    .sort_index()
)

STEP 10C — REPEAT CUSTOMER ANALYSIS

Customer summary:


,customer_type,customer_count,customer_share_%
0,One-Time Customer,90557,97.0
1,Repeat Customer,2801,3.0



Order-count distribution:
delivered_orders
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64


In [39]:
# ============================================================
# STEP 10D — REPEAT PURCHASE INTERVAL
# ============================================================

# Start from delivered orders and include the purchase timestamp
repeat_customers = (
    orders[
        orders["order_status"] == "delivered"
    ][
        ["order_id", "customer_id", "order_purchase_timestamp"]
    ]
    .merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

# Keep only customers with more than one delivered order
repeat_counts = (
    repeat_customers
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

repeat_customer_ids = repeat_counts[
    repeat_counts > 1
].index

repeat_customers = repeat_customers[
    repeat_customers["customer_unique_id"].isin(repeat_customer_ids)
].copy()

# Make sure timestamp is datetime
repeat_customers["order_purchase_timestamp"] = pd.to_datetime(
    repeat_customers["order_purchase_timestamp"],
    errors="coerce"
)

# First and last delivered purchase for each repeat customer
repeat_interval = (
    repeat_customers
    .sort_values(
        ["customer_unique_id", "order_purchase_timestamp"]
    )
    .groupby("customer_unique_id")
    .agg(
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
        order_count=("order_id", "nunique")
    )
    .reset_index()
)

repeat_interval["repeat_purchase_span_days"] = (
    repeat_interval["last_order_date"]
    - repeat_interval["first_order_date"]
).dt.total_seconds() / 86400

print("STEP 10D — REPEAT PURCHASE INTERVAL")

print("\nRepeat customers:", len(repeat_interval))

print("\nRepeat purchase span (days):")
print(
    repeat_interval["repeat_purchase_span_days"].describe()
)

print("\nSample repeat customers:")
display(
    repeat_interval
    .sort_values(
        ["order_count", "repeat_purchase_span_days"],
        ascending=[False, False]
    )
    .head(20)
)

STEP 10D — REPEAT PURCHASE INTERVAL

Repeat customers: 2801

Repeat purchase span (days):
count    2801.000000
mean       88.164526
std       115.367365
min         0.000000
25%         0.007836
50%        34.965845
75%       141.789398
max       633.084190
Name: repeat_purchase_span_days, dtype: float64

Sample repeat customers:


,customer_unique_id,first_order_date,last_order_date,order_count,repeat_purchase_span_days
1551,8d50f5eadf50201ccdcedfb9e2ac8455,2017-06-18 22:56:48,2018-08-20 19:14:26,15,427.845579
670,3e43e6105506432c953e165fb2acf44c,2017-09-18 18:53:15,2018-02-27 18:36:39,9,161.988472
1089,6469f99c1f9dfae7733b25662e7f1782,2017-09-19 01:02:44,2018-06-28 00:43:34,7,281.986690
2233,ca77025e7201e3b30c44b472ff346268,2017-10-09 12:34:39,2018-06-01 11:38:29,7,234.960995
308,1b6c7548a2a1f9037c1fd3ddfed95f33,2017-11-13 16:44:41,2018-02-14 13:22:12,7,92.859387
2426,dc813062e0fc23409cd255f7f53c7074,2017-07-01 04:22:21,2018-08-23 00:07:26,6,417.822975
1078,63cfc61cee11cbe306bff5857d00bfe4,2017-05-11 14:39:53,2018-05-28 17:20:02,6,382.111215
2663,f0e310a6839dce9de1638e0fe5ab282a,2017-05-20 08:53:30,2018-04-05 09:04:45,6,320.007812
767,47c1a3033b8b77b3ab6e109eb4d5fdf3,2017-08-07 14:14:22,2018-01-24 15:15:26,6,170.042407
209,12f5d6e1cbf93dafd9dcc19095df0b3d,2017-01-05 14:18:03,2017-01-05 15:25:10,6,0.046609


In [40]:
# ============================================================
# STEP 10E — ACTUAL INTER-PURCHASE INTERVAL
# ============================================================

repeat_orders = (
    orders[
        orders["order_status"] == "delivered"
    ][
        ["order_id", "customer_id", "order_purchase_timestamp"]
    ]
    .merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

repeat_orders["order_purchase_timestamp"] = pd.to_datetime(
    repeat_orders["order_purchase_timestamp"],
    errors="coerce"
)

repeat_orders = repeat_orders.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

# Calculate gap between consecutive purchases
repeat_orders["days_since_previous_order"] = (
    repeat_orders
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .diff()
    .dt.total_seconds()
    / 86400
)

# Keep only actual repeat purchases
interpurchase = repeat_orders[
    repeat_orders["days_since_previous_order"].notna()
].copy()

print("STEP 10E — ACTUAL INTER-PURCHASE INTERVAL")

print("\nTotal repeat purchase intervals:", len(interpurchase))

print("\nDescriptive statistics (days):")
print(
    interpurchase["days_since_previous_order"].describe()
)

print("\nMedian days between purchases:")
print(
    interpurchase["days_since_previous_order"].median()
)

print("\nIntervals <= 1 day:")
print(
    (interpurchase["days_since_previous_order"] <= 1).sum()
)

print("\nIntervals <= 7 days:")
print(
    (interpurchase["days_since_previous_order"] <= 7).sum()
)

print("\nLargest gaps:")
display(
    interpurchase.sort_values(
        "days_since_previous_order",
        ascending=False
    ).head(10)
)

STEP 10E — ACTUAL INTER-PURCHASE INTERVAL

Total repeat purchase intervals: 3120

Descriptive statistics (days):
count    3120.000000
mean       79.150268
std       107.296346
min         0.000000
25%         0.006620
50%        29.455527
75%       121.468753
max       608.978912
Name: days_since_previous_order, dtype: float64

Median days between purchases:
29.45552662037037

Intervals <= 1 day:
922

Intervals <= 7 days:
1106

Largest gaps:


,order_id,customer_id,order_purchase_timestamp,customer_unique_id,days_since_previous_order
14,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,2018-06-07 19:03:12,ccafc1c3f270410521c3c6f3b249870f,608.978912
40058,b2947cf9a8d3084369dfcbe9663dd545,554e72ada8751c515ab293a7605b530e,2018-08-24 17:52:59,d8f3c4f441a9b59a29f977df16724f38,582.864363
65241,2f53e1db0b2a17564ef12fe69d65a9df,6d334aaab9f45b27efe602b61679ffda,2018-05-09 13:49:19,94e5ea5a8c1bf546db2739673060c43f,580.693322
71274,6d86a700fbdfec909998eefa9110f3e9,6efe2f2c813379b610f5cd26e99ce0ff,2018-05-04 11:14:37,87b3f231705783eb2217e25851c0a45d,572.686840
11191,876fd8c751a56ed982b5a72dca56937f,bb465058eeef2697a5f4a9534e3750e6,2018-03-13 22:28:21,4e23e1826902ec9f208e8cc61329b494,524.413495
54566,57bc9b432f93fd30a3cc83952ccbf267,7022a8efc480a54d4fcaddb10315fa3a,2018-08-25 11:01:56,a1c61f8566347ec44ea37d22854634a1,524.100926
56843,5997512fab61ce978490135668b02aa9,9c159f3aa83cd57f79f8241e0a7f0628,2018-07-24 16:44:34,a262442e3ab89611b44877c7aaf77468,521.928194
1174,c94206a6d6698d5852b7e9e14bdc2e79,0881e1574aa6b621c551a421297af5b8,2018-06-25 12:06:07,18bc87094128bbfe943cf88adcf72059,514.511933
33717,2e8080693faebedb3ba68531167c6313,9e36e87b2ed55c78b24bd7a4b58be3a0,2018-07-14 19:38:26,7e7301841ddb4064c2f3a31e4c154932,514.279468
25455,b196564f2fd79a80ab1b31dd8ba4e4fb,960ffb45acd270534e88387352742459,2018-06-06 21:11:57,24072811917876a84c81166f96aed0c1,510.902106


In [41]:
# ============================================================
# STEP 10F — SAME-DAY REPEAT PURCHASE ANALYSIS
# ============================================================

same_day = interpurchase[
    interpurchase["days_since_previous_order"] == 0
]

print("STEP 10F — SAME-DAY REPEAT PURCHASE ANALYSIS")

print("Same-day repeat intervals:", len(same_day))

print(
    "Percentage of repeat intervals:",
    round(len(same_day) / len(interpurchase) * 100, 2),
    "%"
)

print("\nSmallest non-zero intervals:")
display(
    interpurchase[
        interpurchase["days_since_previous_order"] > 0
    ]
    .sort_values("days_since_previous_order")
    .head(20)
)

STEP 10F — SAME-DAY REPEAT PURCHASE ANALYSIS
Same-day repeat intervals: 267
Percentage of repeat intervals: 8.56 %

Smallest non-zero intervals:


,order_id,customer_id,order_purchase_timestamp,customer_unique_id,days_since_previous_order
14377,92ef3140d9d92187a04b1bd643f32dc1,35852763df46e976c158536d6b99b982,2017-08-28 15:30:28,b7d8524e14913b79fe082a5fc2efffd7,0.000012
40911,60c5f920621dfaf30c3829088dccfb26,52f14b1298aeb699cb2ccfd7d5d2c2de,2017-08-16 02:00:11,b7d4cab5dd11b238da97b01eaf657293,0.000012
58384,d07ea9203ae0a6ea15a2eacc87b4c0f7,2de12a5fa829fedc2d04da8db32bae05,2018-02-15 08:21:05,dc0274a25d881c813da7d55b7faace37,0.000012
66277,46ad8f532d3f76a1747288a6a5a88a18,0924789d2effb3f418c9e8148ae5a5af,2017-10-09 10:54:15,dbdfcd1e9de47452f3c9b0229bc0edc3,0.000012
82951,3d0acef161147c90c2fd5e7cd8e199e1,ee7466b4c80f1bc4f55f2c2ecb0e7c6f,2018-01-14 22:33:42,bcfc1d8cf64a476b003cee4599f9fd11,0.000012
63855,aaa4688b6d2e4d20b2ffc6d97723ea63,827ec074a70a7c08c2b9e7af20bf0287,2018-03-02 09:43:02,fb567adf300e73b60281b72af7702687,0.000012
8573,204c3a076103a93b98207df2faeb6b4c,c908a3e881a6b1e45aba3336b478f445,2017-05-21 08:00:00,fafbc4ecb15933633821dfd530c9263b,0.000012
57547,84f5e6c0a0e3155e38c00f434ba90ce8,0cecd74a34f636f301b750123dd4dd3c,2017-05-06 20:11:11,06a52782a04f0086d16b9c22d0e29438,0.000012
90805,c2cd6ce44609af75d061dfab2f67179c,9ae68689eb558472f387541c67ac8592,2017-05-05 08:44:58,0396c443fdda5498c7e9ed5b34871c5a,0.000012
81282,eda8ab7b9c9d5e8456d446ad54339c2f,9f7e510614e62267e24c6d880a0b1b20,2017-06-29 21:22:47,053f7850c3ffd9726cff61c7e42af797,0.000012


# 7. Payment Behavior

This section examines how customers pay for their orders.

The analysis covers:

- Payment method usage
- Order-level payment penetration
- Payment value
- Installment behavior
- Multiple-payment structures

Payment records are aggregated to the order level where required because a
single order can contain multiple payment records or payment methods.

In [42]:
# ============================================================
# STEP 11 — PAYMENT METHOD ANALYSIS
# ============================================================

payments = data["order_payments"].copy()

payment_summary = (
    payments
    .groupby("payment_type")
    .agg(
        payment_records=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        total_payment_value=("payment_value", "sum"),
        average_payment_value=("payment_value", "mean"),
        median_payment_value=("payment_value", "median"),
        average_installments=("payment_installments", "mean"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

payment_summary["payment_share_%"] = (
    payment_summary["payment_records"]
    / payment_summary["payment_records"].sum()
    * 100
).round(2)

payment_summary = payment_summary.sort_values(
    "total_payment_value",
    ascending=False
)

print("STEP 11 — PAYMENT METHOD ANALYSIS")

display(payment_summary)

STEP 11 — PAYMENT METHOD ANALYSIS


,payment_type,payment_records,unique_orders,total_payment_value,average_payment_value,median_payment_value,average_installments,max_installments,payment_share_%
1,credit_card,76795,76505,12542084.19,163.319021,106.87,3.507155,24,73.92
0,boleto,19784,19784,2869361.27,145.034435,93.89,1.000000,1,19.04
4,voucher,5775,3866,379436.87,65.703354,39.28,1.000000,1,5.56
2,debit_card,1529,1528,217989.79,142.570170,89.30,1.000000,1,1.47
3,not_defined,3,3,0.00,0.000000,0.00,1.000000,1,0.00


In [43]:
# ============================================================
# STEP 11B — PAYMENT METHOD ORDER PENETRATION
# ============================================================

order_payment_summary = (
    payments
    .groupby("payment_type")
    .agg(
        unique_orders=("order_id", "nunique"),
        total_payment_value=("payment_value", "sum")
    )
    .reset_index()
)

total_orders_with_payments = payments["order_id"].nunique()

order_payment_summary["order_penetration_%"] = (
    order_payment_summary["unique_orders"]
    / total_orders_with_payments
    * 100
).round(2)

order_payment_summary = order_payment_summary.sort_values(
    "unique_orders",
    ascending=False
)

print("STEP 11B — PAYMENT METHOD ORDER PENETRATION")

display(order_payment_summary)

STEP 11B — PAYMENT METHOD ORDER PENETRATION


,payment_type,unique_orders,total_payment_value,order_penetration_%
1,credit_card,76505,12542084.19,76.94
0,boleto,19784,2869361.27,19.90
4,voucher,3866,379436.87,3.89
2,debit_card,1528,217989.79,1.54
3,not_defined,3,0.00,0.00


In [44]:
# ============================================================
# STEP 11C — INSTALLMENT ANALYSIS
# ============================================================

installment_summary = (
    payments
    .groupby("payment_installments")
    .agg(
        payment_records=("order_id", "count"),
        unique_orders=("order_id", "nunique"),
        total_payment_value=("payment_value", "sum"),
        average_payment_value=("payment_value", "mean")
    )
    .reset_index()
    .sort_values("payment_installments")
)

installment_summary["payment_record_share_%"] = (
    installment_summary["payment_records"]
    / installment_summary["payment_records"].sum()
    * 100
).round(2)

print("STEP 11C — INSTALLMENT ANALYSIS")

display(installment_summary)

STEP 11C — INSTALLMENT ANALYSIS


,payment_installments,payment_records,unique_orders,total_payment_value,average_payment_value,payment_record_share_%
0,0,2,2,188.63,94.315000,0.00
1,1,52546,49060,5907233.36,112.420229,50.58
2,2,12413,12389,1579283.03,127.228150,11.95
3,3,10461,10443,1491103.80,142.539317,10.07
4,4,7098,7088,1163907.61,163.976840,6.83
5,5,5239,5234,961174.30,183.465222,5.04
6,6,3920,3916,822611.81,209.849952,3.77
7,7,1626,1623,305157.39,187.673672,1.57
8,8,4268,4253,1313423.34,307.737427,4.11
9,9,644,644,131015.92,203.440870,0.62


In [45]:
# Credit-card installment distribution

credit_card_installments = (
    payments[
        payments["payment_type"] == "credit_card"
    ]
    ["payment_installments"]
    .value_counts()
    .sort_index()
)

print("\nCREDIT CARD INSTALLMENT DISTRIBUTION")
print(credit_card_installments)


CREDIT CARD INSTALLMENT DISTRIBUTION
payment_installments
0         2
1     25455
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64


# 8. Delivery Performance & Customer Satisfaction

This section evaluates fulfillment performance and its relationship with
customer satisfaction.

Key metrics include:

- Actual delivery time
- Late-delivery rate
- Delivery performance by customer state
- Review scores for late vs. non-late deliveries

Late delivery is defined as an actual customer-delivery date occurring after
the estimated delivery date.

Observed relationships are reported as associations rather than causal claims.

In [46]:
# ============================================================
# STEP 12 — DELIVERY PERFORMANCE
# ============================================================

delivery = orders[
    orders["order_status"] == "delivered"
].copy()

# Ensure datetime columns are datetime
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    delivery[col] = pd.to_datetime(
        delivery[col],
        errors="coerce"
    )

# Actual delivery time
delivery["delivery_days"] = (
    delivery["order_delivered_customer_date"]
    - delivery["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Days relative to estimated delivery
delivery["delivery_vs_estimate_days"] = (
    delivery["order_delivered_customer_date"].dt.normalize()
    - delivery["order_estimated_delivery_date"].dt.normalize()
).dt.days

# Late delivery flag
delivery["is_late"] = (
    delivery["delivery_vs_estimate_days"] > 0
)

print("STEP 12 — DELIVERY PERFORMANCE")

print("\nDelivered orders:", len(delivery))

print("\nMissing actual delivery dates:")
print(delivery["order_delivered_customer_date"].isna().sum())

print("\nDelivery time (days):")
print(delivery["delivery_days"].describe())

print("\nLate delivery summary:")
print(delivery["is_late"].value_counts())

print("\nLate delivery percentage:")
print(
    round(delivery["is_late"].mean() * 100, 2),
    "%"
)

STEP 12 — DELIVERY PERFORMANCE

Delivered orders: 96478

Missing actual delivery dates:
8

Delivery time (days):
count    96470.000000
mean        12.558217
std          9.546156
min          0.533414
25%          6.766204
50%         10.217477
75%         15.720182
max        209.628611
Name: delivery_days, dtype: float64

Late delivery summary:
is_late
False    89944
True      6534
Name: count, dtype: int64

Late delivery percentage:
6.77 %


In [47]:
# ============================================================
# STEP 12B — DELIVERY PERFORMANCE BY CUSTOMER STATE
# ============================================================

delivery_state = (
    delivery[
        [
            "order_id",
            "customer_id",
            "delivery_days",
            "delivery_vs_estimate_days",
            "is_late"
        ]
    ]
    .merge(
        customers[
            ["customer_id", "customer_state"]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
)

delivery_state_summary = (
    delivery_state
    .groupby("customer_state")
    .agg(
        delivered_orders=("order_id", "nunique"),
        average_delivery_days=("delivery_days", "mean"),
        median_delivery_days=("delivery_days", "median"),
        late_orders=("is_late", "sum"),
        late_delivery_rate=("is_late", "mean")
    )
    .reset_index()
)

delivery_state_summary["late_delivery_rate"] = (
    delivery_state_summary["late_delivery_rate"] * 100
).round(2)

delivery_state_summary = delivery_state_summary.sort_values(
    "late_delivery_rate",
    ascending=False
)

print("STEP 12B — DELIVERY PERFORMANCE BY CUSTOMER STATE")

display(delivery_state_summary)

STEP 12B — DELIVERY PERFORMANCE BY CUSTOMER STATE


,customer_state,delivered_orders,average_delivery_days,median_delivery_days,late_orders,late_delivery_rate
1,AL,397,24.543855,22.332917,85,21.41
9,MA,717,21.572976,19.190602,125,17.43
24,SE,335,21.519788,18.014306,51,15.22
16,PI,476,19.457098,16.292269,66,13.87
5,CE,1279,21.266579,18.209201,176,13.76
21,RR,41,29.387546,25.007303,5,12.20
4,BA,3256,19.335466,16.913964,396,12.16
18,RJ,12350,15.309438,12.041644,1495,12.11
13,PA,946,23.772917,21.078391,106,11.21
7,ES,1995,15.789307,13.636956,214,10.73


In [48]:
# ============================================================
# STEP 12C — RELIABLE DELIVERY PERFORMANCE BY STATE
# Minimum 1,000 delivered orders
# ============================================================

delivery_state_reliable = delivery_state_summary[
    delivery_state_summary["delivered_orders"] >= 1000
].copy()

delivery_state_reliable = delivery_state_reliable.sort_values(
    "late_delivery_rate",
    ascending=False
)

print("STEP 12C — RELIABLE STATE DELIVERY COMPARISON")
print("Minimum delivered orders: 1,000")

display(
    delivery_state_reliable[
        [
            "customer_state",
            "delivered_orders",
            "average_delivery_days",
            "median_delivery_days",
            "late_orders",
            "late_delivery_rate"
        ]
    ]
)

STEP 12C — RELIABLE STATE DELIVERY COMPARISON
Minimum delivered orders: 1,000


,customer_state,delivered_orders,average_delivery_days,median_delivery_days,late_orders,late_delivery_rate
5,CE,1279,21.266579,18.209201,176,13.76
4,BA,3256,19.335466,16.913964,396,12.16
18,RJ,12350,15.309438,12.041644,1495,12.11
7,ES,1995,15.789307,13.636956,214,10.73
15,PE,1593,18.448323,15.702766,153,9.60
23,SC,3546,14.954783,13.014352,291,8.21
8,GO,1957,15.606339,13.949606,128,6.54
22,RS,5345,15.300276,13.177488,325,6.08
6,DF,2080,12.967568,11.364925,118,5.67
10,MG,11354,12.008666,10.313513,519,4.57


In [49]:
# ============================================================
# STEP 12D — DELIVERY VS CUSTOMER SATISFACTION
# ============================================================

# Aggregate reviews to one row per order
review_order_summary = (
    data["order_reviews"]
    .groupby("order_id")
    .agg(
        average_review_score=("review_score", "mean"),
        review_count=("review_id", "nunique")
    )
    .reset_index()
)

# Merge with delivery performance
delivery_reviews = (
    delivery[
        [
            "order_id",
            "delivery_vs_estimate_days",
            "is_late"
        ]
    ]
    .merge(
        review_order_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

# Keep orders with reviews
delivery_reviews = delivery_reviews[
    delivery_reviews["average_review_score"].notna()
].copy()

print("STEP 12D — DELIVERY VS CUSTOMER SATISFACTION")

print("\nOrders with both delivery and review data:",
      len(delivery_reviews))

# Compare average review score
review_by_delivery = (
    delivery_reviews
    .groupby("is_late")
    .agg(
        orders=("order_id", "count"),
        average_review_score=("average_review_score", "mean")
    )
    .reset_index()
)

print("\nReview score by delivery status:")
display(review_by_delivery)

# Review score distribution
print("\nReview score distribution by delivery status:")

review_distribution = pd.crosstab(
    delivery_reviews["is_late"],
    delivery_reviews["average_review_score"]
)

display(review_distribution)

STEP 12D — DELIVERY VS CUSTOMER SATISFACTION

Orders with both delivery and review data: 95832

Review score by delivery status:


,is_late,orders,average_review_score
0,False,89451,4.290608
1,True,6381,2.271823



Review score distribution by delivery status:


average_review_score,1.000000,1.500000,2.000000,2.500000,3.000000,3.333333,3.500000,4.000000,4.333333,4.500000,5.000000
is_late,,,,,,,,,,,
False,5887,7,2364,25,7225,1,22,18217,1,53,55649
True,3426,1,552,5,690,0,1,651,0,0,1055


In [50]:
# ============================================================
# STEP 12E — INDIVIDUAL REVIEW SCORE VS DELIVERY STATUS
# ============================================================

review_scores = (
    data["order_reviews"][
        ["order_id", "review_id", "review_score"]
    ]
    .merge(
        delivery[
            ["order_id", "is_late"]
        ],
        on="order_id",
        how="inner",
        validate="many_to_one"
    )
)

print("STEP 12E — INDIVIDUAL REVIEW SCORE VS DELIVERY STATUS")

review_score_summary = (
    review_scores
    .groupby("is_late")
    .agg(
        review_count=("review_id", "count"),
        average_review_score=("review_score", "mean"),
        median_review_score=("review_score", "median"),
        one_star_reviews=("review_score", lambda x: (x == 1).sum()),
        five_star_reviews=("review_score", lambda x: (x == 5).sum())
    )
    .reset_index()
)

print("\nReview summary:")
display(review_score_summary)

print("\nReview score distribution:")
display(
    pd.crosstab(
        review_scores["is_late"],
        review_scores["review_score"]
    )
)

STEP 12E — INDIVIDUAL REVIEW SCORE VS DELIVERY STATUS

Review summary:


,is_late,review_count,average_review_score,median_review_score,one_star_reviews,five_star_reviews
0,False,89952,4.289999,5.0,5962,56006
1,True,6409,2.271025,1.0,3444,1060



Review score distribution:


review_score,1,2,3,4,5
is_late,,,,,
False,5962,2385,7264,18335,56006
True,3444,556,697,652,1060


# 9. Customer Satisfaction & Category Experience

This section examines review scores across the marketplace and by product
category.

The analysis combines satisfaction with sales scale and delivery performance
to identify categories that may deserve operational investigation.

Minimum review-volume thresholds are applied before comparing category
satisfaction.

In [51]:
# ============================================================
# STEP 13 — OVERALL CUSTOMER SATISFACTION
# ============================================================

reviews = data["order_reviews"]

review_summary = pd.DataFrame({
    "metric": [
        "Total reviews",
        "Average review score",
        "Median review score",
        "1-star reviews",
        "2-star reviews",
        "3-star reviews",
        "4-star reviews",
        "5-star reviews"
    ],
    "value": [
        len(reviews),
        reviews["review_score"].mean(),
        reviews["review_score"].median(),
        (reviews["review_score"] == 1).sum(),
        (reviews["review_score"] == 2).sum(),
        (reviews["review_score"] == 3).sum(),
        (reviews["review_score"] == 4).sum(),
        (reviews["review_score"] == 5).sum()
    ]
})

review_summary["value"] = review_summary["value"].round(2)

print("STEP 13 — OVERALL CUSTOMER SATISFACTION")

display(review_summary)

print("\nReview score distribution:")
display(
    reviews["review_score"]
    .value_counts()
    .sort_index()
)

STEP 13 — OVERALL CUSTOMER SATISFACTION


,metric,value
0,Total reviews,99224.00
1,Average review score,4.09
2,Median review score,5.00
3,1-star reviews,11424.00
4,2-star reviews,3151.00
5,3-star reviews,8179.00
6,4-star reviews,19142.00
7,5-star reviews,57328.00



Review score distribution:


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [52]:
# ============================================================
# STEP 13B — REVIEW SCORE BY PRODUCT CATEGORY
# ============================================================

# Order-level average review score
order_review_score = (
    data["order_reviews"]
    .groupby("order_id")
    .agg(
        average_review_score=("review_score", "mean"),
        review_count=("review_id", "nunique")
    )
    .reset_index()
)

# Unique order-category relationships
order_categories = (
    item_analysis[
        ["order_id", "analysis_category"]
    ]
    .drop_duplicates()
)

# Attach review scores
category_reviews = order_categories.merge(
    order_review_score,
    on="order_id",
    how="inner",
    validate="many_to_one"
)

category_review_summary = (
    category_reviews
    .groupby("analysis_category")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        average_review_score=("average_review_score", "mean"),
        median_review_score=("average_review_score", "median")
    )
    .reset_index()
)

category_review_summary = category_review_summary.sort_values(
    "average_review_score",
    ascending=False
)

print("STEP 13B — REVIEW SCORE BY PRODUCT CATEGORY")

display(category_review_summary)

STEP 13B — REVIEW SCORE BY PRODUCT CATEGORY


,analysis_category,reviewed_orders,average_review_score,median_review_score
11,cds_dvds_musicals,12,4.666667,5.0
29,fashion_childrens_clothes,8,4.500000,5.0
8,books_general_interest,508,4.462598,5.0
22,costruction_tools_tools,94,4.425532,5.0
10,books_technical,257,4.400778,5.0
...,...,...,...,...
30,fashion_male_clothing,111,3.702703,5.0
57,office_furniture,1263,3.616390,4.0
62,portateis_cozinha_e_preparadores_de_alimentos,14,3.428571,3.5
59,pc_gamer,8,3.125000,4.0


In [53]:
# ============================================================
# STEP 13C — RELIABLE CATEGORY REVIEW RANKING
# Minimum reviewed orders: 100
# ============================================================

category_review_reliable = category_review_summary[
    category_review_summary["reviewed_orders"] >= 100
].copy()

category_review_reliable = category_review_reliable.sort_values(
    "average_review_score",
    ascending=False
)

print("STEP 13C — RELIABLE CATEGORY REVIEW RANKING")
print("Minimum reviewed orders: 100")

print("\nTop-rated categories:")
display(
    category_review_reliable.head(10)
)

print("\nLowest-rated categories:")
display(
    category_review_reliable.tail(10)
)

STEP 13C — RELIABLE CATEGORY REVIEW RANKING
Minimum reviewed orders: 100

Top-rated categories:


,analysis_category,reviewed_orders,average_review_score,median_review_score
8,books_general_interest,508,4.462598,5.0
10,books_technical,257,4.400778,5.0
37,food_drink,226,4.382743,5.0
53,luggage_accessories,1030,4.331068,5.0
36,food,445,4.276404,5.0
68,stationery,2295,4.244662,5.0
61,pet_shop,1701,4.238095,5.0
31,fashion_shoes,236,4.218220,5.0
60,perfumery,3150,4.203333,5.0
21,costruction_tools_garden,194,4.190722,5.0



Lowest-rated categories:


,analysis_category,reviewed_orders,average_review_score,median_review_score
7,bed_bath_table,9313,3.970740,5.0
48,home_construction,487,3.967146,5.0
33,fashion_underwear_beach,120,3.933333,4.0
72,uncategorized,1439,3.912787,5.0
34,fixed_telephony,214,3.901869,5.0
47,home_confort,395,3.863291,5.0
19,construction_tools_safety,166,3.849398,5.0
4,audio,347,3.834294,5.0
30,fashion_male_clothing,111,3.702703,5.0
57,office_furniture,1263,3.616390,4.0


In [54]:
# ============================================================
# STEP 13D — CATEGORY SALES VS CUSTOMER SATISFACTION
# ============================================================

category_matrix = (
    category_delivered[
        [
            "analysis_category",
            "total_sales_value",
            "item_count",
            "unique_orders"
        ]
    ]
    .merge(
        category_review_reliable[
            [
                "analysis_category",
                "reviewed_orders",
                "average_review_score"
            ]
        ],
        on="analysis_category",
        how="inner",
        validate="one_to_one"
    )
)

# Rank categories on both dimensions
category_matrix["sales_rank"] = (
    category_matrix["total_sales_value"]
    .rank(method="min", ascending=False)
)

category_matrix["review_rank"] = (
    category_matrix["average_review_score"]
    .rank(method="min", ascending=False)
)

category_matrix = category_matrix.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 13D — CATEGORY SALES VS CUSTOMER SATISFACTION")

display(category_matrix)

STEP 13D — CATEGORY SALES VS CUSTOMER SATISFACTION


,analysis_category,total_sales_value,item_count,unique_orders,reviewed_orders,average_review_score,sales_rank,review_rank
0,health_beauty,1412089.53,9465,8647,8771,4.181203,1.0,15.0
1,watches_gifts,1264333.12,5859,5495,5576,4.065459,2.0,30.0
2,bed_bath_table,1225209.26,10953,9272,9313,3.970740,3.0,43.0
3,sports_leisure,1118256.91,8431,7530,7669,4.167362,4.0,19.0
4,computers_accessories,1032723.77,7644,6530,6649,4.026245,5.0,38.0
5,furniture_decor,880329.92,8160,6307,6398,4.009691,6.0,41.0
6,housewares,758392.25,6795,5743,5843,4.142221,7.0,22.0
7,cool_stuff,691680.89,3718,3559,3599,4.169630,8.0,18.0
8,auto,669454.75,4140,3810,3877,4.090147,9.0,28.0
9,garden_tools,567145.68,4268,3448,3496,4.136156,10.0,23.0


In [55]:
# ============================================================
# STEP 13E — CATEGORY DELIVERY PERFORMANCE VS SATISFACTION
# ============================================================

category_delivery_review = (
    delivery_reviews[
        [
            "order_id",
            "is_late",
            "average_review_score"
        ]
    ]
    .merge(
        item_analysis[
            ["order_id", "analysis_category"]
        ].drop_duplicates(),
        on="order_id",
        how="inner",
        validate="many_to_many"
    )
)

category_delivery_summary = (
    category_delivery_review
    .groupby("analysis_category")
    .agg(
        reviewed_orders=("order_id", "nunique"),
        late_orders=("is_late", "sum"),
        late_delivery_rate=("is_late", "mean"),
        average_review_score=("average_review_score", "mean")
    )
    .reset_index()
)

category_delivery_summary["late_delivery_rate"] = (
    category_delivery_summary["late_delivery_rate"] * 100
).round(2)

category_delivery_summary = category_delivery_summary[
    category_delivery_summary["reviewed_orders"] >= 100
].sort_values(
    "average_review_score"
)

print("STEP 13E — CATEGORY DELIVERY VS SATISFACTION")
print("Minimum reviewed orders: 100")

display(category_delivery_summary)

STEP 13E — CATEGORY DELIVERY VS SATISFACTION
Minimum reviewed orders: 100


,analysis_category,reviewed_orders,late_orders,late_delivery_rate,average_review_score
57,office_furniture,1244,99,7.96,3.642685
30,fashion_male_clothing,105,4,3.81,3.819048
4,audio,345,40,11.59,3.839130
47,home_confort,390,37,9.49,3.887179
34,fixed_telephony,209,9,4.31,3.966507
19,construction_tools_safety,158,4,2.53,3.974684
48,home_construction,481,29,6.03,3.985447
7,bed_bath_table,9177,668,7.28,3.999292
33,fashion_underwear_beach,116,11,9.48,4.008621
72,uncategorized,1382,100,7.24,4.010492


# 10. Customer Geography & Market Concentration

This section examines where customer demand is concentrated.

The analysis covers:

- Customer sales by state
- Customer sales by city
- Sales concentration among top cities

The objective is to distinguish major demand centers from the long tail of
smaller geographic markets.

In [56]:
# ============================================================
# STEP 14 — TOP CUSTOMER CITIES
# ============================================================

customer_city_performance = (
    customer_orders
    .groupby("customer_city")
    .agg(
        unique_customers=("customer_unique_id", "nunique"),
        delivered_orders=("order_id", "nunique"),
        total_sales_value=("sales_value", "sum")
    )
    .reset_index()
)

customer_city_performance["sales_share_%"] = (
    customer_city_performance["total_sales_value"]
    / customer_city_performance["total_sales_value"].sum()
    * 100
).round(2)

customer_city_performance = customer_city_performance.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 14 — TOP CUSTOMER CITIES")

display(
    customer_city_performance.head(20)
)

STEP 14 — TOP CUSTOMER CITIES


,customer_city,unique_customers,delivered_orders,total_sales_value,sales_share_%
3563,sao paulo,14528,15045,2107960.17,13.67
3126,rio de janeiro,6361,6601,1111732.21,7.21
449,belo horizonte,2606,2697,405950.51,2.63
553,brasilia,2013,2071,345199.05,2.24
1135,curitiba,1434,1489,238459.72,1.55
2936,porto alegre,1292,1342,214805.84,1.39
700,campinas,1363,1406,209002.90,1.36
3218,salvador,1154,1188,207713.30,1.35
1518,guarulhos,1111,1144,157735.65,1.02
2440,niteroi,788,825,135447.96,0.88


In [57]:
# ============================================================
# STEP 14B — CUSTOMER CITY CONCENTRATION
# ============================================================

city_concentration = customer_city_performance.sort_values(
    "total_sales_value",
    ascending=False
).copy()

total_city_sales = city_concentration["total_sales_value"].sum()

city_concentration["cumulative_sales_share_%"] = (
    city_concentration["total_sales_value"].cumsum()
    / total_city_sales
    * 100
).round(2)

print("STEP 14B — CUSTOMER CITY CONCENTRATION")

for n in [5, 10, 20]:
    top_n_sales = city_concentration.head(n)["total_sales_value"].sum()
    top_n_share = top_n_sales / total_city_sales * 100
    
    print(
        f"Top {n} cities: "
        f"{top_n_sales:,.2f} sales value "
        f"({top_n_share:.2f}%)"
    )

print("\nTop 20 cities:")
display(
    city_concentration[
        [
            "customer_city",
            "unique_customers",
            "delivered_orders",
            "total_sales_value",
            "sales_share_%",
            "cumulative_sales_share_%"
        ]
    ].head(20)
)

STEP 14B — CUSTOMER CITY CONCENTRATION
Top 5 cities: 4,209,301.66 sales value (27.30%)
Top 10 cities: 5,134,007.31 sales value (33.29%)
Top 20 cities: 6,169,319.62 sales value (40.01%)

Top 20 cities:


,customer_city,unique_customers,delivered_orders,total_sales_value,sales_share_%,cumulative_sales_share_%
3563,sao paulo,14528,15045,2107960.17,13.67,13.67
3126,rio de janeiro,6361,6601,1111732.21,7.21,20.88
449,belo horizonte,2606,2697,405950.51,2.63,23.51
553,brasilia,2013,2071,345199.05,2.24,25.75
1135,curitiba,1434,1489,238459.72,1.55,27.30
2936,porto alegre,1292,1342,214805.84,1.39,28.69
700,campinas,1363,1406,209002.90,1.36,30.05
3218,salvador,1154,1188,207713.30,1.35,31.39
1518,guarulhos,1111,1144,157735.65,1.02,32.42
2440,niteroi,788,825,135447.96,0.88,33.29


In [58]:
# ============================================================
# STEP 15 — CORE BUSINESS KPI SUMMARY
# ============================================================

total_orders = len(orders)

delivered_order_count = (
    orders["order_status"] == "delivered"
).sum()

total_customers = customers["customer_unique_id"].nunique()

repeat_customer_count = (
    customer_order_counts["customer_type"]
    .eq("Repeat Customer")
    .sum()
)

one_time_customer_count = (
    customer_order_counts["customer_type"]
    .eq("One-Time Customer")
    .sum()
)

total_sales_value = delivered_sales["sales_value"].sum()

average_order_value = (
    total_sales_value / delivered_order_count
)

average_review_score = reviews["review_score"].mean()

late_delivery_rate = delivery["is_late"].mean() * 100

credit_card_order_penetration = (
    order_payment_summary.loc[
        order_payment_summary["payment_type"] == "credit_card",
        "order_penetration_%"
    ].iloc[0]
)

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Delivered Orders",
        "Delivery Rate (%)",
        "Unique Customers",
        "One-Time Customers",
        "Repeat Customers",
        "Repeat Customer Rate (%)",
        "Delivered Sales Value (incl. freight)",
        "Average Order Value",
        "Average Review Score",
        "Late Delivery Rate (%)",
        "Credit Card Order Penetration (%)"
    ],
    "Value": [
        total_orders,
        delivered_order_count,
        delivered_order_count / total_orders * 100,
        total_customers,
        one_time_customer_count,
        repeat_customer_count,
        repeat_customer_count / total_customers * 100,
        total_sales_value,
        average_order_value,
        average_review_score,
        late_delivery_rate,
        credit_card_order_penetration
    ]
})

kpi_summary["Value"] = kpi_summary["Value"].round(2)

print("STEP 15 — CORE BUSINESS KPI SUMMARY")

display(kpi_summary)

STEP 15 — CORE BUSINESS KPI SUMMARY


,KPI,Value
0,Total Orders,99441.00
1,Delivered Orders,96478.00
2,Delivery Rate (%),97.02
3,Unique Customers,96096.00
4,One-Time Customers,90557.00
5,Repeat Customers,2801.00
6,Repeat Customer Rate (%),2.91
7,Delivered Sales Value (incl. freight),15419773.75
8,Average Order Value,159.83
9,Average Review Score,4.09


# 11. Building the Order-Level Analytical Dataset

The analysis now moves from independent exploratory tables to an integrated
order-level analytical dataset.

Target grain:

> One row = one order

One-to-many tables such as order items, payments, and reviews are aggregated
to one row per order before being joined to the main `orders` table.

This prevents row multiplication and ensures that order-level KPIs such as
sales, payment value, and review metrics are calculated correctly.

In [59]:
# ============================================================
# STEP 16A — ORDER ITEM SUMMARY
# ONE ROW PER ORDER
# ============================================================

order_item_summary = (
    data["order_items"]
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        product_sales=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
    .reset_index()
)

order_item_summary["sales_value_incl_freight"] = (
    order_item_summary["product_sales"]
    + order_item_summary["freight_value"]
)

print("STEP 16A — ORDER ITEM SUMMARY")

print("Rows:", len(order_item_summary))
print("Unique orders:", order_item_summary["order_id"].nunique())

display(order_item_summary.head(10))

STEP 16A — ORDER ITEM SUMMARY
Rows: 98666
Unique orders: 98666


,order_id,item_count,unique_products,unique_sellers,product_sales,freight_value,sales_value_incl_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14,218.04
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1,1,21.90,12.69,34.59
6,00054e8431b9d7675808bcb819fb4a32,1,1,1,19.90,11.85,31.75
7,000576fe39319847cbb9d288c5617fa6,1,1,1,810.00,70.75,880.75
8,0005a1a1728c9d785b8e2b08b904576c,1,1,1,145.95,11.65,157.60
9,0005f50442cb953dcd1d21e1fb923495,1,1,1,53.99,11.40,65.39


In [60]:
# ============================================================
# STEP 16B — ORDER PAYMENT SUMMARY
# ONE ROW PER ORDER
# ============================================================

order_payment_summary = (
    data["order_payments"]
    .groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        unique_payment_types=("payment_type", "nunique"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

print("STEP 16B — ORDER PAYMENT SUMMARY")

print("Rows:", len(order_payment_summary))
print(
    "Unique orders:",
    order_payment_summary["order_id"].nunique()
)

display(order_payment_summary.head(10))

STEP 16B — ORDER PAYMENT SUMMARY
Rows: 99440
Unique orders: 99440


,order_id,payment_record_count,total_payment_value,unique_payment_types,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,1,3
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,34.59,1,1
6,00054e8431b9d7675808bcb819fb4a32,1,31.75,1,1
7,000576fe39319847cbb9d288c5617fa6,1,880.75,1,10
8,0005a1a1728c9d785b8e2b08b904576c,1,157.60,1,3
9,0005f50442cb953dcd1d21e1fb923495,1,65.39,1,1


In [61]:
# ============================================================
# STEP 16C — ORDER REVIEW SUMMARY
# ONE ROW PER ORDER
# ============================================================

order_review_summary = (
    data["order_reviews"]
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        unique_review_ids=("review_id", "nunique"),
        average_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max")
    )
    .reset_index()
)

print("STEP 16C — ORDER REVIEW SUMMARY")

print("Rows:", len(order_review_summary))
print(
    "Unique orders:",
    order_review_summary["order_id"].nunique()
)

print("\nOrders with multiple reviews:")

display(
    order_review_summary[
        order_review_summary["review_count"] > 1
    ].head(10)
)

print("\nSummary preview:")

display(order_review_summary.head(10))

STEP 16C — ORDER REVIEW SUMMARY
Rows: 98673
Unique orders: 98673

Orders with multiple reviews:


,order_id,review_count,unique_review_ids,average_review_score,min_review_score,max_review_score
84,0035246a40f520710769010f752e7507,2,2,5.000000,5,5
461,013056cfe49763c6f66bda03396c5ee3,2,2,4.500000,4,5
556,0176a6846bcb3b0d3aa3116a9a768597,2,2,5.000000,5,5
835,02355020fd0a40a0d56df9f6ff060413,2,2,2.000000,1,3
985,029863af4b968de1e5d6a82782e662f5,2,2,4.500000,4,5
1092,02e0b68852217f5715fb9cc885829454,2,2,4.000000,4,4
1103,02e723e8edb4a123d414f56cc9c4665e,2,2,5.000000,5,5
1272,03515a836bb855b03f7df9dee520a8fc,2,2,5.000000,5,5
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,3,3.333333,3,4
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,2,4.500000,4,5



Summary preview:


,order_id,review_count,unique_review_ids,average_review_score,min_review_score,max_review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,5.0,5,5
1,00018f77f2f0320c557190d7a144bdd3,1,1,4.0,4,4
2,000229ec398224ef6ca0657da4fc703e,1,1,5.0,5,5
3,00024acbcdf0a6daa1e931b038114c75,1,1,4.0,4,4
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,5.0,5,5
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1,4.0,4,4
6,00054e8431b9d7675808bcb819fb4a32,1,1,4.0,4,4
7,000576fe39319847cbb9d288c5617fa6,1,1,5.0,5,5
8,0005a1a1728c9d785b8e2b08b904576c,1,1,1.0,1,1
9,0005f50442cb953dcd1d21e1fb923495,1,1,4.0,4,4


In [62]:
# ============================================================
# STEP 16D — MASTER ORDER-LEVEL ANALYTICAL TABLE
# ONE ROW PER ORDER
# ============================================================

master_orders = (
    orders.copy()
    .merge(
        customers[
            [
                "customer_id",
                "customer_unique_id",
                "customer_city",
                "customer_state"
            ]
        ],
        on="customer_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        order_item_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        order_payment_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        order_review_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

print("STEP 16D — MASTER ORDER-LEVEL ANALYTICAL TABLE")

print("\nRows:", len(master_orders))
print("Unique order IDs:", master_orders["order_id"].nunique())

print("\nExpected rows:", len(orders))

print("\nDuplicate order IDs:")
print(
    master_orders["order_id"].duplicated().sum()
)

print("\nShape:", master_orders.shape)

print("\nMissing values in aggregated fields:")
display(
    master_orders[
        [
            "item_count",
            "total_payment_value",
            "average_review_score"
        ]
    ].isna().sum()
)

print("\nPreview:")
display(master_orders.head(10))

STEP 16D — MASTER ORDER-LEVEL ANALYTICAL TABLE

Rows: 99441
Unique order IDs: 99441

Expected rows: 99441

Duplicate order IDs:
0

Shape: (99441, 27)

Missing values in aggregated fields:


item_count              775
total_payment_value       1
average_review_score    768
dtype: int64


Preview:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_month,customer_unique_id,...,sales_value_incl_freight,payment_record_count,total_payment_value,unique_payment_types,max_installments,review_count,unique_review_ids,average_review_score,min_review_score,max_review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,7c396fd4830fd04220f754e42b4e5bff,...,38.71,3.0,38.71,2.0,1.0,1.0,1.0,4.0,4.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-07,af07308b275d755c9edb36a90c618231,...,141.46,1.0,141.46,1.0,1.0,1.0,1.0,4.0,4.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08,3a653a41f6f9fc3d2a113cf8398680e8,...,179.12,1.0,179.12,1.0,3.0,1.0,1.0,5.0,5.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,2017-11,7c142cf63193a1473d2e66489a9ae977,...,72.20,1.0,72.20,1.0,1.0,1.0,1.0,5.0,5.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2018-02,72632f0f9dd73dfee390c9b22eb56dd6,...,28.62,1.0,28.62,1.0,1.0,1.0,1.0,5.0,5.0,5.0
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,2017-07,80bb27c7c16e8f973207a5086ab329e2,...,175.26,1.0,175.26,1.0,6.0,1.0,1.0,4.0,4.0,4.0
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaN,NaN,2017-05-09,2017-04,36edbb3fb164b1f16485364b6fb04c73,...,65.95,1.0,65.95,1.0,1.0,1.0,1.0,2.0,2.0,2.0
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,2017-05,932afa1e708222e5821dac9cd5db4cae,...,75.16,1.0,75.16,1.0,3.0,1.0,1.0,5.0,5.0,5.0
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,2017-01,39382392765b6dc74812866ee5ee92a7,...,35.95,1.0,35.95,1.0,1.0,1.0,1.0,1.0,1.0,1.0
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,2017-07,299905e3934e9e181bfb2e164dd4b4f8,...,169.76,2.0,169.76,2.0,1.0,1.0,1.0,5.0,5.0,5.0


In [63]:
# ============================================================
# STEP 16E — ORDER-LEVEL DERIVED METRICS
# ============================================================

# Make sure datetime columns are datetime
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in datetime_cols:
    master_orders[col] = pd.to_datetime(
        master_orders[col],
        errors="coerce"
    )

# Order month
master_orders["order_month"] = (
    master_orders["order_purchase_timestamp"]
    .dt.to_period("M")
)

# Delivered flag
master_orders["is_delivered"] = (
    master_orders["order_status"] == "delivered"
)

# Delivery days
master_orders["delivery_days"] = (
    master_orders["order_delivered_customer_date"]
    - master_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Difference from estimated delivery
master_orders["delivery_vs_estimate_days"] = (
    master_orders["order_delivered_customer_date"].dt.normalize()
    - master_orders["order_estimated_delivery_date"].dt.normalize()
).dt.days

# Late flag
master_orders["is_late"] = (
    master_orders["is_delivered"]
    & master_orders["delivery_vs_estimate_days"].notna()
    & (master_orders["delivery_vs_estimate_days"] > 0)
)

# Review available flag
master_orders["has_review"] = (
    master_orders["average_review_score"].notna()
)

# Payment available flag
master_orders["has_payment"] = (
    master_orders["total_payment_value"].notna()
)

# Item available flag
master_orders["has_items"] = (
    master_orders["item_count"].notna()
)

print("STEP 16E — ORDER-LEVEL DERIVED METRICS")

print("\nShape:", master_orders.shape)

print("\nDelivered orders:", master_orders["is_delivered"].sum())

print("Late delivered orders:", master_orders["is_late"].sum())

print("Orders with reviews:", master_orders["has_review"].sum())

print("Orders with payments:", master_orders["has_payment"].sum())

print("Orders with items:", master_orders["has_items"].sum())

print("\nNew columns:")
print([
    "order_month",
    "is_delivered",
    "delivery_days",
    "delivery_vs_estimate_days",
    "is_late",
    "has_review",
    "has_payment",
    "has_items"
])

STEP 16E — ORDER-LEVEL DERIVED METRICS

Shape: (99441, 34)

Delivered orders: 96478
Late delivered orders: 6534
Orders with reviews: 98673
Orders with payments: 99440
Orders with items: 98666

New columns:
['order_month', 'is_delivered', 'delivery_days', 'delivery_vs_estimate_days', 'is_late', 'has_review', 'has_payment', 'has_items']


In [64]:
# ============================================================
# STEP 16F — SAVE MASTER ORDER TABLE
# ============================================================

from pathlib import Path

ANALYSIS_DATA_PATH = Path("../Data/Analysis")
ANALYSIS_DATA_PATH.mkdir(parents=True, exist_ok=True)

master_orders_path = (
    ANALYSIS_DATA_PATH / "master_orders.csv"
)

master_orders.to_csv(
    master_orders_path,
    index=False
)

print("STEP 16F — MASTER ORDER TABLE SAVED")

print("Saved to:", master_orders_path)
print("Rows:", len(master_orders))
print("Columns:", len(master_orders.columns))

STEP 16F — MASTER ORDER TABLE SAVED
Saved to: ..\Data\Analysis\master_orders.csv
Rows: 99441
Columns: 34


# 12. Customer Value & Retention

This section evaluates the economic importance of repeat customers.

The analysis compares one-time and repeat customers and examines whether
higher purchase frequency is associated with higher customer value.

A scenario analysis is then used to illustrate the potential scale of a
second-purchase opportunity.

Scenario values are illustrative and are not revenue forecasts.

In [65]:
# ============================================================
# STEP 17 — CUSTOMER VALUE BY CUSTOMER TYPE
# ============================================================

customer_type_value = (
    master_orders[
        master_orders["is_delivered"]
        & master_orders["customer_unique_id"].notna()
    ]
    .groupby("customer_unique_id")
    .agg(
        delivered_orders=("order_id", "nunique"),
        total_customer_sales=("sales_value_incl_freight", "sum")
    )
    .reset_index()
)

customer_type_value["customer_type"] = (
    customer_type_value["delivered_orders"]
    .apply(
        lambda x: "Repeat Customer"
        if x > 1
        else "One-Time Customer"
    )
)

customer_type_summary = (
    customer_type_value
    .groupby("customer_type")
    .agg(
        customer_count=("customer_unique_id", "count"),
        total_orders=("delivered_orders", "sum"),
        total_sales_value=("total_customer_sales", "sum"),
        average_sales_per_customer=("total_customer_sales", "mean"),
        median_sales_per_customer=("total_customer_sales", "median")
    )
    .reset_index()
)

customer_type_summary["sales_share_%"] = (
    customer_type_summary["total_sales_value"]
    / customer_type_summary["total_sales_value"].sum()
    * 100
).round(2)

print("STEP 17 — CUSTOMER VALUE BY CUSTOMER TYPE")

display(customer_type_summary.round(2))

STEP 17 — CUSTOMER VALUE BY CUSTOMER TYPE


,customer_type,customer_count,total_orders,total_sales_value,average_sales_per_customer,median_sales_per_customer,sales_share_%
0,One-Time Customer,90557,90557,14555586.29,160.73,105.38,94.4
1,Repeat Customer,2801,5921,864187.46,308.53,225.55,5.6


In [66]:
# ============================================================
# STEP 17B — REPEAT CUSTOMER VALUE BY ORDER FREQUENCY
# ============================================================

repeat_value = customer_type_value[
    customer_type_value["customer_type"] == "Repeat Customer"
].copy()

repeat_value["frequency_group"] = pd.cut(
    repeat_value["delivered_orders"],
    bins=[1, 2, 3, float("inf")],
    labels=["2 Orders", "3 Orders", "4+ Orders"],
    include_lowest=True
)

repeat_frequency_summary = (
    repeat_value
    .groupby("frequency_group", observed=False)
    .agg(
        customer_count=("customer_unique_id", "count"),
        total_orders=("delivered_orders", "sum"),
        total_sales_value=("total_customer_sales", "sum"),
        average_sales_per_customer=("total_customer_sales", "mean"),
        median_sales_per_customer=("total_customer_sales", "median")
    )
    .reset_index()
)

repeat_frequency_summary["customer_share_%"] = (
    repeat_frequency_summary["customer_count"]
    / repeat_frequency_summary["customer_count"].sum()
    * 100
).round(2)

repeat_frequency_summary["sales_share_%"] = (
    repeat_frequency_summary["total_sales_value"]
    / repeat_frequency_summary["total_sales_value"].sum()
    * 100
).round(2)

print("STEP 17B — REPEAT CUSTOMER VALUE BY ORDER FREQUENCY")

display(repeat_frequency_summary.round(2))

STEP 17B — REPEAT CUSTOMER VALUE BY ORDER FREQUENCY


,frequency_group,customer_count,total_orders,total_sales_value,average_sales_per_customer,median_sales_per_customer,customer_share_%,sales_share_%
0,2 Orders,2573,5146,748811.57,291.03,218.94,91.86,86.65
1,3 Orders,181,543,78333.95,432.78,331.11,6.46,9.06
2,4+ Orders,47,232,37041.94,788.13,646.99,1.68,4.29


In [67]:
# ============================================================
# STEP 17C — SECOND-PURCHASE CONVERSION RATE
# ============================================================

total_customers = customer_type_value["customer_unique_id"].nunique()

customers_with_second_order = (
    customer_type_value["delivered_orders"] >= 2
).sum()

second_purchase_rate = (
    customers_with_second_order
    / total_customers
    * 100
)

print("STEP 17C — SECOND-PURCHASE CONVERSION RATE")

print("Total customers:", total_customers)
print("Customers with 2+ orders:", customers_with_second_order)
print(
    "Second-purchase conversion rate:",
    round(second_purchase_rate, 2),
    "%"
)

STEP 17C — SECOND-PURCHASE CONVERSION RATE
Total customers: 93358
Customers with 2+ orders: 2801
Second-purchase conversion rate: 3.0 %


In [68]:
# ============================================================
# STEP 18 — CUSTOMER COHORT RETENTION
# ============================================================

cohort_data = (
    master_orders[
        master_orders["is_delivered"]
        & master_orders["customer_unique_id"].notna()
    ][
        [
            "customer_unique_id",
            "order_id",
            "order_purchase_timestamp"
        ]
    ]
    .copy()
)

cohort_data["order_month"] = (
    pd.to_datetime(cohort_data["order_purchase_timestamp"])
    .dt.to_period("M")
)

# First delivered order month for each customer
first_purchase = (
    cohort_data
    .groupby("customer_unique_id")["order_month"]
    .min()
    .rename("cohort_month")
)

cohort_data = cohort_data.merge(
    first_purchase,
    on="customer_unique_id",
    how="left",
    validate="many_to_one"
)

# Cohort age in months
cohort_data["cohort_index"] = (
    (cohort_data["order_month"].dt.year - cohort_data["cohort_month"].dt.year) * 12
    + (cohort_data["order_month"].dt.month - cohort_data["cohort_month"].dt.month)
)

cohort_counts = (
    cohort_data
    .groupby(
        ["cohort_month", "cohort_index"]
    )["customer_unique_id"]
    .nunique()
    .reset_index(name="active_customers")
)

cohort_size = (
    cohort_data
    .groupby("cohort_month")["customer_unique_id"]
    .nunique()
    .rename("cohort_size")
    .reset_index()
)

cohort_counts = cohort_counts.merge(
    cohort_size,
    on="cohort_month",
    how="left"
)

cohort_counts["retention_%"] = (
    cohort_counts["active_customers"]
    / cohort_counts["cohort_size"]
    * 100
).round(2)

print("STEP 18 — CUSTOMER COHORT RETENTION")

print("\nCohort sizes:")
display(cohort_size.head(20))

print("\nRetention data:")
display(cohort_counts.head(30))

STEP 18 — CUSTOMER COHORT RETENTION

Cohort sizes:


,cohort_month,cohort_size
0,2016-09,1
1,2016-10,262
2,2016-12,1
3,2017-01,717
4,2017-02,1628
5,2017-03,2503
6,2017-04,2256
7,2017-05,3451
8,2017-06,3037
9,2017-07,3752



Retention data:


,cohort_month,cohort_index,active_customers,cohort_size,retention_%
0,2016-09,0,1,1,100.00
1,2016-10,0,262,262,100.00
2,2016-10,6,1,262,0.38
3,2016-10,9,1,262,0.38
4,2016-10,11,1,262,0.38
5,2016-10,13,1,262,0.38
6,2016-10,15,1,262,0.38
7,2016-10,17,1,262,0.38
8,2016-10,19,2,262,0.76
9,2016-10,20,2,262,0.76


In [69]:
# ============================================================
# STEP 18B — STANDARD COHORT RETENTION MATRIX
# ============================================================

# Number of unique customers in each cohort at each cohort age
cohort_activity = (
    cohort_data
    .groupby(
        ["cohort_month", "cohort_index"]
    )["customer_unique_id"]
    .nunique()
    .reset_index(name="active_customers")
)

# Cohort sizes
cohort_sizes = (
    cohort_data
    .groupby("cohort_month")["customer_unique_id"]
    .nunique()
)

# Convert active customers into retention %
cohort_activity["retention_%"] = (
    cohort_activity["active_customers"]
    / cohort_activity["cohort_month"].map(cohort_sizes)
    * 100
)

# Pivot to a proper retention matrix
retention_matrix = (
    cohort_activity
    .pivot(
        index="cohort_month",
        columns="cohort_index",
        values="retention_%"
    )
)

# Keep only useful milestones
milestones = [0, 1, 2, 3, 6, 12]

available_milestones = [
    m for m in milestones
    if m in retention_matrix.columns
]

retention_milestones = retention_matrix[
    available_milestones
].copy()

retention_milestones.columns = [
    f"month_{int(col)}"
    for col in retention_milestones.columns
]

print("STEP 18B — COHORT RETENTION MILESTONES")

display(
    retention_milestones.round(2)
)

STEP 18B — COHORT RETENTION MILESTONES


,month_0,month_1,month_2,month_3,month_6,month_12
cohort_month,,,,,,
2016-09,100.0,NaN,NaN,NaN,NaN,NaN
2016-10,100.0,NaN,NaN,NaN,0.38,NaN
2016-12,100.0,100.00,NaN,NaN,NaN,NaN
2017-01,100.0,0.28,0.28,0.14,0.42,0.70
2017-02,100.0,0.18,0.31,0.12,0.25,0.12
2017-03,100.0,0.44,0.36,0.40,0.16,0.20
2017-04,100.0,0.62,0.22,0.18,0.35,0.04
2017-05,100.0,0.46,0.46,0.29,0.41,0.23
2017-06,100.0,0.49,0.40,0.43,0.36,0.16


In [70]:
# ============================================================
# STEP 18C — OVERALL COHORT RETENTION BY MILESTONE
# ============================================================

milestone_summary = []

for month in available_milestones:

    # Only cohorts old enough to have reached this month
    eligible = retention_matrix[
        retention_matrix.index <= (
            retention_matrix.index.max() - month
        )
    ][month]

    milestone_summary.append({
        "cohort_month_index": month,
        "eligible_cohorts": eligible.notna().sum(),
        "average_retention_%": eligible.mean(),
        "median_retention_%": eligible.median()
    })

milestone_summary = pd.DataFrame(milestone_summary)

print("STEP 18C — OVERALL COHORT RETENTION")

display(
    milestone_summary.round(2)
)

STEP 18C — OVERALL COHORT RETENTION


,cohort_month_index,eligible_cohorts,average_retention_%,median_retention_%
0,0,23,100.00,100.00
1,1,20,5.45,0.51
2,2,18,0.34,0.33
3,3,17,0.25,0.27
4,6,15,0.27,0.25
5,12,8,0.21,0.15


# 13. Order Economics & High-Value Orders

This section examines the distribution of delivered order values.

The objective is to determine whether sales are broadly distributed or
concentrated among a relatively small number of high-value orders.

The analysis also identifies customers with high-value transactions using
the 99th percentile as a data-driven threshold.

In [71]:
# ============================================================
# STEP 19 — ORDER VALUE DISTRIBUTION
# ============================================================

order_value_analysis = master_orders[
    master_orders["is_delivered"]
    & master_orders["sales_value_incl_freight"].notna()
].copy()

print("STEP 19 — ORDER VALUE DISTRIBUTION")

print("\nDescriptive statistics:")
display(
    order_value_analysis["sales_value_incl_freight"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nTotal delivered sales:")
print(
    round(
        order_value_analysis["sales_value_incl_freight"].sum(),
        2
    )
)

STEP 19 — ORDER VALUE DISTRIBUTION

Descriptive statistics:


count    96478.000000
mean       159.826839
std        218.794219
min          9.590000
25%         61.850000
50%        105.280000
75%        176.260000
90%        305.918000
95%        446.232000
99%       1052.389900
max      13664.080000
Name: sales_value_incl_freight, dtype: float64


Total delivered sales:
15419773.75


In [72]:
# ============================================================
# STEP 19B — HIGH-VALUE ORDER CONTRIBUTION
# ============================================================

values = order_value_analysis[
    "sales_value_incl_freight"
].sort_values(ascending=False)

total_sales = values.sum()

top_1_pct_count = max(1, int(len(values) * 0.01))
top_5_pct_count = max(1, int(len(values) * 0.05))
top_10_pct_count = max(1, int(len(values) * 0.10))

top_1_sales = values.head(top_1_pct_count).sum()
top_5_sales = values.head(top_5_pct_count).sum()
top_10_sales = values.head(top_10_pct_count).sum()

print("STEP 19B — HIGH-VALUE ORDER CONTRIBUTION")

print(
    f"Top 1% of orders sales share: "
    f"{top_1_sales / total_sales * 100:.2f}%"
)

print(
    f"Top 5% of orders sales share: "
    f"{top_5_sales / total_sales * 100:.2f}%"
)

print(
    f"Top 10% of orders sales share: "
    f"{top_10_sales / total_sales * 100:.2f}%"
)

STEP 19B — HIGH-VALUE ORDER CONTRIBUTION
Top 1% of orders sales share: 10.39%
Top 5% of orders sales share: 26.72%
Top 10% of orders sales share: 38.05%


In [73]:
# ============================================================
# STEP 19C — HIGH-VALUE CUSTOMER SEGMENT
# ============================================================

high_value_threshold = (
    order_value_analysis["sales_value_incl_freight"]
    .quantile(0.99)
)

high_value_orders = order_value_analysis[
    order_value_analysis["sales_value_incl_freight"]
    >= high_value_threshold
].copy()

high_value_customer_summary = (
    high_value_orders
    .groupby("customer_unique_id")
    .agg(
        high_value_orders=("order_id", "count"),
        high_value_sales=("sales_value_incl_freight", "sum"),
        average_high_value_order=("sales_value_incl_freight", "mean")
    )
    .reset_index()
    .sort_values(
        "high_value_sales",
        ascending=False
    )
)

print("STEP 19C — HIGH-VALUE CUSTOMER SEGMENT")

print(
    "High-value order threshold (99th percentile):",
    round(high_value_threshold, 2)
)

print(
    "High-value orders:",
    len(high_value_orders)
)

print(
    "Customers with at least one high-value order:",
    high_value_customer_summary["customer_unique_id"].nunique()
)

print("\nTop high-value customers:")
display(
    high_value_customer_summary.head(20)
)

STEP 19C — HIGH-VALUE CUSTOMER SEGMENT
High-value order threshold (99th percentile): 1052.39
High-value orders: 965
Customers with at least one high-value order: 960

Top high-value customers:


,customer_unique_id,high_value_orders,high_value_sales,average_high_value_order
43,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.080
813,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.815
453,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.880
817,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.310
264,459bef486812aa25204be022145caa62,1,6922.21,6922.210
955,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.660
243,4007669dec559734d6f53e029e360987,1,6081.54,6081.540
880,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.340
277,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.780
871,edde2314c6c30e864a128ac95d6b2112,1,4513.32,4513.320


# 14. Marketing Funnel Analysis

The marketing dataset contains 8,000 marketing-qualified leads (MQLs), of
which 842 are linked to closed deals.

This section evaluates:

- Overall MQL-to-closed-deal conversion
- Conversion by lead origin
- Composition of successful deals by lead type
- Business segment
- Lead behaviour profile
- Business type

Where a descriptive attribute exists only in the closed-deal dataset, the
analysis reports composition of successful deals rather than claiming a
conversion rate.

In [74]:
# ============================================================
# STEP 20 — MARKETING LEAD → CLOSED DEAL COVERAGE
# ============================================================

mql = data["marketing_qualified_leads"].copy()
closed = data["closed_deals"].copy()

print("STEP 20 — MARKETING LEAD → CLOSED DEAL COVERAGE")

# Unique MQL IDs
mql_ids = set(mql["mql_id"].dropna())
closed_mql_ids = set(closed["mql_id"].dropna())

# Closed deals whose MQL does not exist in the MQL table
invalid_closed_mql = closed_mql_ids - mql_ids

# MQLs that became closed deals
matched_mqls = mql_ids & closed_mql_ids

print("\nTotal MQL records:", len(mql))
print("Unique MQL IDs:", mql["mql_id"].nunique())

print("\nTotal closed deals:", len(closed))
print("Unique closed MQL IDs:", closed["mql_id"].nunique())

print("\nClosed deals with missing MQL record:",
      len(invalid_closed_mql))

print("\nMQLs that became closed deals:",
      len(matched_mqls))

print(
    "\nOverall MQL → closed-deal conversion rate:",
    round(
        len(matched_mqls) / mql["mql_id"].nunique() * 100,
        2
    ),
    "%"
)

STEP 20 — MARKETING LEAD → CLOSED DEAL COVERAGE

Total MQL records: 8000
Unique MQL IDs: 8000

Total closed deals: 842
Unique closed MQL IDs: 842

Closed deals with missing MQL record: 0

MQLs that became closed deals: 842

Overall MQL → closed-deal conversion rate: 10.53 %


In [75]:
# ============================================================
# STEP 20B — MQL CONVERSION BY ORIGIN
# ============================================================

closed_mql_set = set(
    closed["mql_id"].dropna()
)

origin_conversion = (
    mql
    .groupby("origin", dropna=False)
    .agg(
        total_mqls=("mql_id", "nunique")
    )
    .reset_index()
)

# Count closed deals within each origin
closed_by_origin = (
    closed
    .merge(
        mql[["mql_id", "origin"]],
        on="mql_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("origin", dropna=False)
    .agg(
        closed_deals=("mql_id", "nunique")
    )
    .reset_index()
)

origin_conversion = origin_conversion.merge(
    closed_by_origin,
    on="origin",
    how="left"
)

origin_conversion["closed_deals"] = (
    origin_conversion["closed_deals"]
    .fillna(0)
    .astype(int)
)

origin_conversion["conversion_rate_%"] = (
    origin_conversion["closed_deals"]
    / origin_conversion["total_mqls"]
    * 100
).round(2)

origin_conversion = origin_conversion.sort_values(
    "conversion_rate_%",
    ascending=False
)

print("STEP 20B — MQL CONVERSION BY ORIGIN")

display(origin_conversion)

STEP 20B — MQL CONVERSION BY ORIGIN


,origin,total_mqls,closed_deals,conversion_rate_%
10,NaN,60,14,23.33
9,unknown,1099,179,16.29
6,paid_search,1586,195,12.30
3,organic_search,2296,271,11.80
0,direct_traffic,499,56,11.22
7,referral,284,24,8.45
8,social,1350,75,5.56
1,display,118,6,5.08
5,other_publicities,65,3,4.62
2,email,493,15,3.04


In [76]:
# ============================================================
# STEP 20C — LEAD TYPE ANALYSIS AMONG CLOSED DEALS
# ============================================================

print("STEP 20C — LEAD TYPE ANALYSIS AMONG CLOSED DEALS")

print("\nClosed deals shape:", closed.shape)

print("\nClosed deals columns:")
print(closed.columns.tolist())

print("\nLead type distribution:")
display(
    closed["lead_type"]
    .value_counts(dropna=False)
    .rename_axis("lead_type")
    .reset_index(name="closed_deals")
)

STEP 20C — LEAD TYPE ANALYSIS AMONG CLOSED DEALS

Closed deals shape: (842, 15)

Closed deals columns:
['mql_id', 'seller_id', 'sdr_id', 'sr_id', 'won_date', 'business_segment', 'lead_type', 'lead_behaviour_profile', 'has_company', 'has_gtin', 'average_stock', 'business_type', 'declared_product_catalog_size', 'declared_monthly_revenue', 'average_stock_level']

Lead type distribution:


,lead_type,closed_deals
0,online_medium,332
1,online_big,126
2,industry,123
3,offline,104
4,online_small,77
5,online_beginner,57
6,online_top,14
7,NaN,6
8,other,3


In [77]:
# ============================================================
# STEP 20D — CLOSED DEALS BY BUSINESS SEGMENT
# ============================================================

business_segment_summary = (
    closed["business_segment"]
    .value_counts(dropna=False)
    .rename_axis("business_segment")
    .reset_index(name="closed_deals")
)

business_segment_summary["closed_deal_share_%"] = (
    business_segment_summary["closed_deals"]
    / len(closed)
    * 100
).round(2)

business_segment_summary = business_segment_summary.sort_values(
    "closed_deals",
    ascending=False
)

print("STEP 20D — CLOSED DEALS BY BUSINESS SEGMENT")

display(business_segment_summary)

STEP 20D — CLOSED DEALS BY BUSINESS SEGMENT


,business_segment,closed_deals,closed_deal_share_%
0,home_decor,105,12.47
1,health_beauty,93,11.05
2,car_accessories,77,9.14
3,household_utilities,71,8.43
4,construction_tools_house_garden,69,8.19
5,audio_video_electronics,64,7.60
6,computers,34,4.04
7,pet,30,3.56
8,food_supplement,28,3.33
9,food_drink,26,3.09


In [78]:
# ============================================================
# STEP 20E — CLOSED DEALS BY LEAD BEHAVIOUR PROFILE
# ============================================================

lead_behaviour_summary = (
    closed["lead_behaviour_profile"]
    .value_counts(dropna=False)
    .rename_axis("lead_behaviour_profile")
    .reset_index(name="closed_deals")
)

lead_behaviour_summary["closed_deal_share_%"] = (
    lead_behaviour_summary["closed_deals"]
    / len(closed)
    * 100
).round(2)

print("STEP 20E — CLOSED DEALS BY LEAD BEHAVIOUR PROFILE")

display(lead_behaviour_summary)

STEP 20E — CLOSED DEALS BY LEAD BEHAVIOUR PROFILE


,lead_behaviour_profile,closed_deals,closed_deal_share_%
0,cat,407,48.34
1,NaN,177,21.02
2,eagle,123,14.61
3,wolf,95,11.28
4,shark,24,2.85
5,"cat, wolf",8,0.95
6,"eagle, wolf",3,0.36
7,"eagle, cat",3,0.36
8,"shark, cat",1,0.12
9,"shark, wolf",1,0.12


In [79]:
# ============================================================
# STEP 20F — CLOSED DEALS BY BUSINESS TYPE
# ============================================================

business_type_summary = (
    closed["business_type"]
    .value_counts(dropna=False)
    .rename_axis("business_type")
    .reset_index(name="closed_deals")
)

business_type_summary["closed_deal_share_%"] = (
    business_type_summary["closed_deals"]
    / len(closed)
    * 100
).round(2)

print("STEP 20F — CLOSED DEALS BY BUSINESS TYPE")

display(business_type_summary)

STEP 20F — CLOSED DEALS BY BUSINESS TYPE


,business_type,closed_deals,closed_deal_share_%
0,reseller,587,69.71
1,manufacturer,242,28.74
2,NaN,10,1.19
3,other,3,0.36


# 15. Freight Burden & Operational Economics

True profitability cannot be calculated from the available datasets because
product cost, seller payouts, platform fees, and other operating costs are not
available.

Therefore, this section uses freight burden as an operational economics
proxy:

Freight Burden % = Freight Value / Product Sales × 100

The overall marketplace benchmark is used to identify high-volume categories
with above-average freight pressure.

In [80]:
# ============================================================
# STEP 21A — FREIGHT BURDEN BY PRODUCT CATEGORY
# ============================================================

category_economics = (
    category_delivered[
        [
            "analysis_category",
            "item_count",
            "product_sales",
            "freight_value",
            "total_sales_value"
        ]
    ]
    .copy()
)

category_economics["freight_to_product_sales_%"] = (
    category_economics["freight_value"]
    / category_economics["product_sales"]
    * 100
)

category_economics["freight_to_product_sales_%"] = (
    category_economics["freight_to_product_sales_%"]
    .round(2)
)

category_economics = category_economics.sort_values(
    "total_sales_value",
    ascending=False
)

print("STEP 21A — FREIGHT BURDEN BY PRODUCT CATEGORY")

display(category_economics)

STEP 21A — FREIGHT BURDEN BY PRODUCT CATEGORY


,analysis_category,item_count,product_sales,freight_value,total_sales_value,freight_to_product_sales_%
43,health_beauty,9465,1233131.72,178957.81,1412089.53,14.51
73,watches_gifts,5859,1166176.98,98156.14,1264333.12,8.42
7,bed_bath_table,10953,1023434.76,201774.50,1225209.26,19.72
67,sports_leisure,8431,954852.55,163404.36,1118256.91,17.11
15,computers_accessories,7644,888724.61,143999.16,1032723.77,16.20
...,...,...,...,...,...,...
59,pc_gamer,8,1306.95,123.15,1430.10,9.42
46,home_comfort_2,30,760.27,410.31,1170.58,53.97
11,cds_dvds_musicals,14,730.00,224.99,954.99,30.82
29,fashion_childrens_clothes,7,519.95,78.72,598.67,15.14


In [81]:
# ============================================================
# STEP 21B — HIGH-SCALE CATEGORIES WITH FREIGHT PRESSURE
# Minimum delivered items: 2,000
# ============================================================

category_economics_reliable = category_economics[
    category_economics["item_count"] >= 2000
].copy()

category_economics_reliable = (
    category_economics_reliable
    .sort_values(
        "freight_to_product_sales_%",
        ascending=False
    )
)

print("STEP 21B — HIGH-SCALE CATEGORIES WITH FREIGHT PRESSURE")
print("Minimum delivered items: 2,000")

display(
    category_economics_reliable[
        [
            "analysis_category",
            "item_count",
            "product_sales",
            "freight_value",
            "total_sales_value",
            "freight_to_product_sales_%"
        ]
    ]
)

STEP 21B — HIGH-SCALE CATEGORIES WITH FREIGHT PRESSURE
Minimum delivered items: 2,000


,analysis_category,item_count,product_sales,freight_value,total_sales_value,freight_to_product_sales_%
26,electronics,2729,155043.93,45679.16,200723.09,29.46
39,furniture_decor,8160,711927.69,168402.23,880329.92,23.65
49,housewares,6795,615628.69,142763.56,758392.25,23.19
70,telephony,4430,309860.23,69342.39,379202.62,22.38
42,garden_tools,4268,470495.28,96650.40,567145.68,20.54
68,stationery,2466,223788.69,45786.36,269575.05,20.46
7,bed_bath_table,10953,1023434.76,201774.50,1225209.26,19.72
67,sports_leisure,8431,954852.55,163404.36,1118256.91,17.11
6,baby,2982,400421.84,66305.81,466727.65,16.56
15,computers_accessories,7644,888724.61,143999.16,1032723.77,16.20


In [82]:
# ============================================================
# STEP 21C — OVERALL FREIGHT BURDEN BENCHMARK
# ============================================================

overall_product_sales = category_economics["product_sales"].sum()
overall_freight_value = category_economics["freight_value"].sum()
overall_sales_value = category_economics["total_sales_value"].sum()

overall_freight_ratio = (
    overall_freight_value
    / overall_product_sales
    * 100
)

print("STEP 21C — OVERALL FREIGHT BURDEN BENCHMARK")

print("Overall product sales:", round(overall_product_sales, 2))
print("Overall freight value:", round(overall_freight_value, 2))
print("Overall sales value incl. freight:", round(overall_sales_value, 2))
print("Overall freight / product sales:", round(overall_freight_ratio, 2), "%")

STEP 21C — OVERALL FREIGHT BURDEN BENCHMARK
Overall product sales: 13221498.11
Overall freight value: 2198275.64
Overall sales value incl. freight: 15419773.75
Overall freight / product sales: 16.63 %


# 16. Business Priorities & Scenario Analysis

The previous sections identified the main performance patterns.

This section converts those findings into business priorities and scenario
analyses.

The goal is not to generate additional descriptive statistics, but to
quantify the scale of the most important opportunities and identify areas
where management attention may have the highest potential impact.

In [83]:
# ============================================================
# STEP 22A — RETENTION OPPORTUNITY SCENARIO
# ============================================================

one_time_customers = customer_type_value[
    customer_type_value["customer_type"] == "One-Time Customer"
].copy()

repeat_two_order = repeat_value[
    repeat_value["delivered_orders"] == 2
].copy()

observed_two_order_value_per_customer = (
    repeat_two_order["total_customer_sales"].mean()
)

print("STEP 22A — RETENTION OPPORTUNITY SCENARIO")

print(
    "Observed average sales/customer among 2-order customers:",
    round(observed_two_order_value_per_customer, 2)
)

for conversion_rate in [1, 3, 5, 10]:

    customers_converted = (
        len(one_time_customers) * conversion_rate / 100
    )

    potential_sales = (
        customers_converted
        * observed_two_order_value_per_customer
    )

    print(
        f"\nIf {conversion_rate}% of one-time customers "
        f"became 2-order customers:"
    )
    print(
        "Customers converted:",
        round(customers_converted)
    )
    print(
        "Illustrative sales value:",
        round(potential_sales, 2)
    )

STEP 22A — RETENTION OPPORTUNITY SCENARIO
Observed average sales/customer among 2-order customers: 291.03

If 1% of one-time customers became 2-order customers:
Customers converted: 906
Illustrative sales value: 263545.0

If 3% of one-time customers became 2-order customers:
Customers converted: 2717
Illustrative sales value: 790635.01

If 5% of one-time customers became 2-order customers:
Customers converted: 4528
Illustrative sales value: 1317725.02

If 10% of one-time customers became 2-order customers:
Customers converted: 9056
Illustrative sales value: 2635450.03


In [84]:
# ============================================================
# STEP 22B — DELIVERY IMPROVEMENT SCENARIO
# ============================================================

late_order_count = int(delivery["is_late"].sum())
delivered_order_count = len(delivery)

print("STEP 22B — DELIVERY IMPROVEMENT SCENARIO")

print("Current late orders:", late_order_count)
print(
    "Current late-delivery rate:",
    round(late_order_count / delivered_order_count * 100, 2),
    "%"
)

for improvement_rate in [25, 50, 75, 100]:

    late_orders_reduced = (
        late_order_count * improvement_rate / 100
    )

    remaining_late_orders = (
        late_order_count - late_orders_reduced
    )

    new_late_rate = (
        remaining_late_orders
        / delivered_order_count
        * 100
    )

    print(f"\nIf {improvement_rate}% of late deliveries were eliminated:")
    print(
        "Late orders avoided:",
        round(late_orders_reduced)
    )
    print(
        "Remaining late-delivery rate:",
        round(new_late_rate, 2),
        "%"
    )

STEP 22B — DELIVERY IMPROVEMENT SCENARIO
Current late orders: 6534
Current late-delivery rate: 6.77 %

If 25% of late deliveries were eliminated:
Late orders avoided: 1634
Remaining late-delivery rate: 5.08 %

If 50% of late deliveries were eliminated:
Late orders avoided: 3267
Remaining late-delivery rate: 3.39 %

If 75% of late deliveries were eliminated:
Late orders avoided: 4900
Remaining late-delivery rate: 1.69 %

If 100% of late deliveries were eliminated:
Late orders avoided: 6534
Remaining late-delivery rate: 0.0 %


In [85]:
# ============================================================
# STEP 22C — BUSINESS PRIORITY FRAMEWORK
# ============================================================

priority_framework = pd.DataFrame({
    "business_issue": [
        "Low repeat purchase / retention",
        "Late delivery",
        "High freight burden in major categories",
        "Category satisfaction weakness",
        "Marketing attribution / channel quality"
    ],
    "evidence": [
        "Only ~3% repeat customers; repeat customers generate ~1.92x sales/customer",
        "6.77% late; late orders average 2.27 vs 4.29 review score",
        "Overall freight burden 16.63%; several high-scale categories exceed 20%",
        "Office furniture ~3.64; bed_bath_table ~4.00",
        "Overall MQL-to-closed conversion 10.53%; strong variation by origin"
    ],
    "business_impact": [
        "High",
        "High",
        "Medium-High",
        "Medium-High",
        "Medium"
    ],
    "actionability": [
        "High",
        "High",
        "Medium-High",
        "Medium",
        "Medium"
    ],
    "priority": [
        1,
        1,
        2,
        2,
        3
    ]
})

print("STEP 22C — BUSINESS PRIORITY FRAMEWORK")

display(priority_framework)

STEP 22C — BUSINESS PRIORITY FRAMEWORK


,business_issue,evidence,business_impact,actionability,priority
0,Low repeat purchase / retention,Only ~3% repeat customers; repeat customers ge...,High,High,1
1,Late delivery,6.77% late; late orders average 2.27 vs 4.29 r...,High,High,1
2,High freight burden in major categories,Overall freight burden 16.63%; several high-sc...,Medium-High,Medium-High,2
3,Category satisfaction weakness,Office furniture ~3.64; bed_bath_table ~4.00,Medium-High,Medium,2
4,Marketing attribution / channel quality,Overall MQL-to-closed conversion 10.53%; stron...,Medium,Medium,3


# 17. Final Business Insights & Recommendations

The analysis is now synthesized into a small set of actionable business
findings.

Recommendations are based on observed patterns in the dataset and are
distinguished from assumptions, scenario estimates, and causal claims.

In [86]:
# ============================================================
# STEP 23 — FINAL BUSINESS INSIGHTS
# ============================================================

final_insights = pd.DataFrame({
    "priority": [
        1,
        1,
        2,
        2,
        3
    ],

    "insight": [
        "Customer retention is very low",
        "Late delivery is strongly associated with poor satisfaction",
        "Freight burden is significant in several high-volume categories",
        "Some high-volume categories combine weak satisfaction with operational pressure",
        "Marketing conversion varies materially by lead origin"
    ],

    "key_metric": [
        "2.91% repeat customer rate",
        "2.27 vs 4.29 average review score",
        "16.63% overall freight/product-sales ratio",
        "Office furniture ~3.64 review score; bed_bath_table ~19.72% freight burden",
        "10.53% overall MQL-to-closed conversion"
    ],

    "business_recommendation": [
        "Prioritize second-purchase campaigns and retention initiatives",
        "Target delivery bottlenecks and reduce late orders",
        "Optimize freight and fulfillment for high-scale categories above benchmark",
        "Investigate product/service issues in low-rated high-volume categories",
        "Improve channel attribution and prioritize measurable high-performing sources"
    ]
})

print("STEP 23 — FINAL BUSINESS INSIGHTS")

display(final_insights)

STEP 23 — FINAL BUSINESS INSIGHTS


,priority,insight,key_metric,business_recommendation
0,1,Customer retention is very low,2.91% repeat customer rate,Prioritize second-purchase campaigns and reten...
1,1,Late delivery is strongly associated with poor...,2.27 vs 4.29 average review score,Target delivery bottlenecks and reduce late or...
2,2,Freight burden is significant in several high-...,16.63% overall freight/product-sales ratio,Optimize freight and fulfillment for high-scal...
3,2,Some high-volume categories combine weak satis...,Office furniture ~3.64 review score; bed_bath_...,Investigate product/service issues in low-rate...
4,3,Marketing conversion varies materially by lead...,10.53% overall MQL-to-closed conversion,Improve channel attribution and prioritize mea...


# 18. Final KPI Audit & Quality Assurance

Before publishing the analysis, all headline KPIs are reconciled against the
underlying datasets.

This final audit verifies:

- Row counts
- Customer denominator definitions
- Delivered-order counts
- Sales value
- AOV
- Delivery performance
- Review metrics
- Payment penetration
- Freight burden
- MQL conversion

The purpose is to ensure that the numbers used in the dashboard, README,
and presentation are internally consistent.

In [87]:
# ============================================================
# STEP 24 — FINAL KPI CONSISTENCY & QA AUDIT
# ============================================================

# -----------------------------
# Customer population
# -----------------------------

delivered_customer_base = customer_type_value[
    "customer_unique_id"
].nunique()

repeat_customers = (
    customer_type_value["delivered_orders"] >= 2
).sum()

repeat_rate = (
    repeat_customers
    / delivered_customer_base
    * 100
)

# -----------------------------
# Sales / order KPIs
# -----------------------------

delivered_orders = master_orders[
    master_orders["is_delivered"]
].copy()

delivered_order_count = len(delivered_orders)

delivered_sales = (
    delivered_orders["sales_value_incl_freight"]
    .sum()
)

aov = (
    delivered_sales
    / delivered_order_count
)

# -----------------------------
# Delivery KPI
# -----------------------------

valid_delivery = delivered_orders[
    delivered_orders["delivery_days"].notna()
].copy()

late_orders = int(
    valid_delivery["is_late"].sum()
)

late_rate = (
    late_orders
    / len(valid_delivery)
    * 100
)

# -----------------------------
# Review KPI
# -----------------------------

review_count = len(data["order_reviews"])

average_review = (
    data["order_reviews"]["review_score"].mean()
)

# -----------------------------
# Payment KPI
# -----------------------------

payment_orders = (
    data["order_payments"]["order_id"]
    .nunique()
)

credit_card_orders = (
    data["order_payments"]
    .loc[
        data["order_payments"]["payment_type"]
        == "credit_card",
        "order_id"
    ]
    .nunique()
)

credit_card_penetration = (
    credit_card_orders
    / payment_orders
    * 100
)

# -----------------------------
# Freight KPI
# -----------------------------

overall_product_sales = (
    category_economics["product_sales"].sum()
)

overall_freight = (
    category_economics["freight_value"].sum()
)

freight_ratio = (
    overall_freight
    / overall_product_sales
    * 100
)

# -----------------------------
# Marketing KPI
# -----------------------------

mql_count = mql["mql_id"].nunique()
closed_deal_count = closed["mql_id"].nunique()

mql_conversion = (
    closed_deal_count
    / mql_count
    * 100
)

# -----------------------------
# Final KPI table
# -----------------------------

final_kpi_audit = pd.DataFrame({
    "KPI": [
        "Total Orders",
        "Delivered Orders",
        "Unique Delivered Customers",
        "Repeat Customers",
        "Repeat Customer Rate (%)",
        "Delivered Sales Value incl. Freight",
        "Average Order Value",
        "Late Delivery Rate (%)",
        "Total Reviews",
        "Average Review Score",
        "Orders with Payment Records",
        "Credit Card Order Penetration (%)",
        "Freight / Product Sales (%)",
        "MQLs",
        "Closed Deals",
        "MQL → Closed Deal Conversion (%)"
    ],

    "Value": [
        len(master_orders),
        delivered_order_count,
        delivered_customer_base,
        repeat_customers,
        repeat_rate,
        delivered_sales,
        aov,
        late_rate,
        review_count,
        average_review,
        payment_orders,
        credit_card_penetration,
        freight_ratio,
        mql_count,
        closed_deal_count,
        mql_conversion
    ],

    "Definition": [
        "All orders in orders table",
        "Orders with order_status = delivered",
        "Unique customer_unique_id among delivered-order customers",
        "Customers with at least 2 delivered orders",
        "Repeat customers / unique delivered customers",
        "Product sales + freight for delivered orders",
        "Delivered sales value / delivered orders",
        "Late delivered orders / delivered orders with valid delivery dates",
        "Rows in order_reviews",
        "Mean review_score",
        "Unique orders represented in order_payments",
        "Orders using credit_card / orders with payment records",
        "Freight value / product sales",
        "Unique MQL IDs",
        "Unique MQL IDs present in closed_deals",
        "Closed deals / total MQLs"
    ]
})

final_kpi_audit["Value"] = final_kpi_audit["Value"].round(2)

print("STEP 24 — FINAL KPI CONSISTENCY & QA AUDIT")

display(final_kpi_audit)

STEP 24 — FINAL KPI CONSISTENCY & QA AUDIT


,KPI,Value,Definition
0,Total Orders,99441.00,All orders in orders table
1,Delivered Orders,96478.00,Orders with order_status = delivered
2,Unique Delivered Customers,93358.00,Unique customer_unique_id among delivered-orde...
3,Repeat Customers,2801.00,Customers with at least 2 delivered orders
4,Repeat Customer Rate (%),3.00,Repeat customers / unique delivered customers
5,Delivered Sales Value incl. Freight,15419773.75,Product sales + freight for delivered orders
6,Average Order Value,159.83,Delivered sales value / delivered orders
7,Late Delivery Rate (%),6.77,Late delivered orders / delivered orders with ...
8,Total Reviews,99224.00,Rows in order_reviews
9,Average Review Score,4.09,Mean review_score
